## New York ComStock Data Analysis ##

~~Seperately, but related to the overall California AC to Heat Pump work is an effort to clean, merge, and summarize the Commercial Building Stock (ComStock) data in order to provide helpful insights for our data analysis layers. The code below runs in a few separated blocks to deliver interim and final data steps along with checks on our work!~~

__Key Areas/Code Blocks Below Include:__
- ~~(1) Package & data imports to kick off the process~~
- ~~(2) A web scraper designed to pull the NREL/NLR ComStock data by county, organized & delivered in the local file folders (Or you can have the results from Step 3 serve as the base)~~
- ~~(3.1 & 3) A massive merger code block that combines the counties into a singular, large Excel file~~
    - ~~(WARNING! This is computationally intensive to run, do not be surprised if it takes 30+ minutes*!)~~
- ~~(4) The first version of an interactive map to display the ComStock data focused on displaying tonnage distribution by unit typology using a Leaflet HTML mapping style ~~
    - ~~(WARNING! This is computationally intensive to run, less so than the previous but still ~5 minutes!)~~
- ~~(5) The second version of this interactive map is focused on grouping HVAC tonnage distribution by unit type, so we can isolate/elevate the most significant players in the space. The data is further grouped by building type so we can understand where transition impact can be focused~~
    - ~~(WARNING! This is computationally intensive to run, again about ~5 minutes on a fast machine!)~~
- ~~(6) The heart of this final product, using the census tract data in the NLR data set to model the estimated impacts of transition down to the most localizable level, using the conversion estimates derived from the [TRC study](https://www.pinole.gov/wp-content/uploads/2025/12/2025-NR-Alterations-CostEff-Report-1.pdf), adjusted by climate zone, building vintage, and utility provider~~


__NY Sources:__

- (ZZZ) [State Boundaries](https://gis.ny.gov/civil-boundaries) used for county level, specifically eliminating waterways from the plotting areas (necessary for the large number of NY counties that contain channels, islands, etc.)
- (ZZZ) [Climate Zone Boundaries](https://basc.pnnl.gov/guide-determining-climate-zone-county-data-files) using the national dataset to backup the local data in case ComStock has any gaps. Per the previous iteration in California, we need this to fill in for about 15% of cases.
- (ZZZ) 

_Bonus Content:_
- (7) By special request, a county-level rollup of the data results from step (6) was created that allows for a toggle to fleixbly convert different estimated levels of relevant systems. For this data analysis, the TRC study values only addressed Retail and Small Office AC to HP values, however for this toggle this goes further than the step 6 analysis, which assumes full conversion. This specific option allows for a toggle to view the potential for limited scale conversions. Set to 30% as the default since the TRC study indicates that ~30% of HVAC units of the relevant type are 5-20 ton units, the current legislative efforts in CA are focused on addressing that specific market, however future efforts or amendments to the current work could result in more or lesser scale conversion impacts, leading to a need for a toggle.


*_For reference this code was written & run on a MacBook Pro M4 with 16gb of RAM when I stopped all unnecessary other activities to free up memory, so the timing estimates may vary._

In [1]:
# (1) Package import for web scraping
# ==============================================================================
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

import json
import sys
import glob
import os
import time
import warnings
import requests
import xlsxwriter

import folium
from folium import plugins, Choropleth

import selenium
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
warnings.filterwarnings('ignore') # You will want to turn this off!

# Shapefile import and setting the correct CRS for the census tracts
census_tracts_gdf = gpd.read_file('Data Sources/NY_tl_2022_36_tract.zip')
census_tracts_gdf = census_tracts_gdf.to_crs("EPSG:4269")
print(census_tracts_gdf.head(5))

# Checking the census tract shapefile to see what columns are available for merging with the census data
# census_tracts_gdf.head()
# Checking the census crs
# census_tracts_gdf.crs

  STATEFP COUNTYFP TRACTCE        GEOID    NAME             NAMELSAD  MTFCC  \
0      36      007  012702  36007012702  127.02  Census Tract 127.02  G5020   
1      36      007  012800  36007012800     128     Census Tract 128  G5020   
2      36      007  012900  36007012900     129     Census Tract 129  G5020   
3      36      007  013000  36007013000     130     Census Tract 130  G5020   
4      36      007  013202  36007013202  132.02  Census Tract 132.02  G5020   

  FUNCSTAT     ALAND  AWATER     INTPTLAT      INTPTLON  \
0        S  65461841  222705  +42.0350532  -075.9055509   
1        S  12342848  259435  +42.1298743  -075.9096569   
2        S  14480163   63649  +42.1522758  -075.9766029   
3        S   9934434  381729  +42.1236499  -076.0002197   
4        S   2446208    3681  +42.1238924  -076.0311921   

                                            geometry  
0  POLYGON ((-75.95917 42.00852, -75.95914 42.008...  
1  POLYGON ((-75.94766 42.13727, -75.94453 42.137...  
2  PO

In [2]:
import os
import requests
import pandas as pd

# (2) Web Scraping & Data Harvesting from OEDI's data repository of NREL/NLR ComStock data
# ==============================================================================
# Configuration section (please update DEST_FOLDER to your local machine path, check the URL_TEMPLATE if code fails)
DEST_FOLDER = '/Users/neil.stein/Documents/CFB/Western Campaigns/NYSERDA_AC2HP/Data Sources/COMStock'
URL_TEMPLATE = "https://oedi-data-lake.s3.amazonaws.com/nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2025/comstock_amy2018_release_3/metadata_and_annual_results/by_state_and_county/full/csv/state=NY/county={gisjoin}/NY_{gisjoin}_upgrade{upgrade}.csv.gz"

# Explicitly target ONLY these exact upgrade scenarios
TARGET_UPGRADES = [0, 1, 2, 4]

# Dictionary mapping GISJOIN codes for NY Counties to their county names
ny_counties_gisjoin = {
    "Albany": "G3600010",
    "Allegany": "G3600030",
    "Bronx": "G3600050",
    "Broome": "G3600070",
    "Cattaraugus": "G3600090",
    "Cayuga": "G3600110",
    "Chautauqua": "G3600130",
    "Chemung": "G3600150",
    "Chenango": "G3600170",
    "Clinton": "G3600190",
    "Columbia": "G3600210",
    "Cortland": "G3600230",
    "Delaware": "G3600250",
    "Dutchess": "G3600270",
    "Erie": "G3600290",
    "Essex": "G3600310",
    "Franklin": "G3600330",
    "Fulton": "G3600350",
    "Genesee": "G3600370",
    "Greene": "G3600390",
    "Hamilton": "G3600410",
    "Herkimer": "G3600430",
    "Jefferson": "G3600450",
    "Kings": "G3600470",
    "Lewis": "G3600490",
    "Livingston": "G3600510",
    "Madison": "G3600530",
    "Monroe": "G3600550",
    "Montgomery": "G3600570",
    "Nassau": "G3600590",
    "New York": "G3600610",
    "Niagara": "G3600630",
    "Oneida": "G3600650",
    "Onondaga": "G3600670",
    "Ontario": "G3600690",
    "Orange": "G3600710",
    "Orleans": "G3600730",
    "Oswego": "G3600750",
    "Otsego": "G3600770",
    "Putnam": "G3600790",
    "Queens": "G3600810",
    "Rensselaer": "G3600830",
    "Richmond": "G3600850",
    "Rockland": "G3600870",
    "St. Lawrence": "G3600890",
    "Saratoga": "G3600910",
    "Schenectady": "G3600930",
    "Schoharie": "G3600950",
    "Schuyler": "G3600970",
    "Seneca": "G3600990",
    "Steuben": "G3601010",
    "Suffolk": "G3601030",
    "Sullivan": "G3601050",
    "Tioga": "G3601070",
    "Tompkins": "G3601090",
    "Ulster": "G3601110",
    "Warren": "G3601130",
    "Washington": "G3601150",
    "Wayne": "G3601170",
    "Westchester": "G3601190",
    "Wyoming": "G3601210",
    "Yates": "G3601230"
}


# ==============================================================================
# Function for downloading and decompressing data into .csv machine readable files for each county
def download_and_decompress():
    if not os.path.exists(DEST_FOLDER):
        os.makedirs(DEST_FOLDER, exist_ok=True)
        
    total_files = len(ny_counties_gisjoin) * len(TARGET_UPGRADES)
    print(f"--- Phase 1: Downloading & Decompressing {total_files} Files ---")
    
    for county, gisjoin in ny_counties_gisjoin.items():
        for upgrade in TARGET_UPGRADES:
            # Inject variables directly into the template in the configuration section
            url = URL_TEMPLATE.format(gisjoin=gisjoin, upgrade=upgrade)
            
            # Label files strictly as <County Name>_<upgrade scenario>
            file_label = f"{county}_upgrade{upgrade}"
            gz_path = os.path.join(DEST_FOLDER, f"{file_label}.csv.gz")
            final_csv = os.path.join(DEST_FOLDER, f"{file_label}.csv")
            
            if os.path.exists(final_csv):
                print(f"  [v] {file_label}.csv already exists.")
                continue
                
            print(f"  [ ] Downloading and Decompressing {file_label}...")
            try:
                r = requests.get(url, stream=True, timeout=60)
                if r.status_code == 200:
                    # Save temp gz file
                    with open(gz_path, 'wb') as f:
                        for chunk in r.iter_content(chunk_size=1024*1024):
                            f.write(chunk)
                    
                    # Convert to .csv
                    df = pd.read_csv(gz_path, compression='gzip', low_memory=False)
                    df.to_csv(final_csv, index=False)
                    
                    # Remove .gz
                    os.remove(gz_path)
                else:
                    print(f"      [!] Failed: HTTP {r.status_code}")
            except Exception as e:
                print(f"      [!] Error on {file_label}: {e}")

if __name__ == "__main__":
    download_and_decompress()

--- Phase 1: Downloading & Decompressing 248 Files ---
  [ ] Downloading and Decompressing Albany_upgrade0...
  [ ] Downloading and Decompressing Albany_upgrade1...
  [ ] Downloading and Decompressing Albany_upgrade2...
  [ ] Downloading and Decompressing Albany_upgrade4...
  [ ] Downloading and Decompressing Allegany_upgrade0...
  [ ] Downloading and Decompressing Allegany_upgrade1...
  [ ] Downloading and Decompressing Allegany_upgrade2...
  [ ] Downloading and Decompressing Allegany_upgrade4...
  [ ] Downloading and Decompressing Bronx_upgrade0...
  [ ] Downloading and Decompressing Bronx_upgrade1...
  [ ] Downloading and Decompressing Bronx_upgrade2...
  [ ] Downloading and Decompressing Bronx_upgrade4...
  [ ] Downloading and Decompressing Broome_upgrade0...
  [ ] Downloading and Decompressing Broome_upgrade1...
  [ ] Downloading and Decompressing Broome_upgrade2...
  [ ] Downloading and Decompressing Broome_upgrade4...
  [ ] Downloading and Decompressing Cattaraugus_upgrade0...
 

In [4]:
import os
import pandas as pd
import xlsxwriter

# (3) Full format now that we have proven this works
# ==============================================================================
# Configuration section (please update SOURCE_DIR to your local machine path you set in the previous block)
SOURCE_DIR = '/Users/neil.stein/Documents/CFB/Western Campaigns/NYSERDA_AC2HP/Data Sources/COMStock'
ROW_LIMIT = 1048575 # max row count for Excel so it doesn't over-index itself

# Explicitly target ONLY these exact upgrade scenarios for master files
TARGET_UPGRADES = [0, 1, 2, 4]


def build_full_master_excel():
    # Verify the directory exists
    if not os.path.exists(SOURCE_DIR):
        print(f"Error: Source directory {SOURCE_DIR} does not exist.")
        return

    # Loop through each target upgrade scenario to build separate Master Excel files
    for upgrade in TARGET_UPGRADES:
        print(f"\n==============================================================================")
        print(f"Processing Scenario: Upgrade {upgrade}")
        print(f"==============================================================================")
        
        # Explicitly filter for CSV files matching the precise upgrade suffix (e.g., "_upgrade0.csv")
        # This prevents collisions with multi-digit upgrades (like upgrade 14 when looking for 1 or 4)
        target_suffix = f"_upgrade{upgrade}.csv"
        files = sorted([f for f in os.listdir(SOURCE_DIR) if f.endswith(target_suffix)])
        
        if not files:
            print(f"Warning: No CSV files found in {SOURCE_DIR} matching '{target_suffix}'")
            continue

        # Define the exact file name for this specific upgrade master spreadsheet
        final_excel_path = os.path.join(SOURCE_DIR, f"NY_Master_COMStock_Upgrade{upgrade}_FINAL_COMPLETE.xlsx")
        print(f"--- Starting Master Excel Build with {len(files)} Counties for Upgrade {upgrade} ---")
        
        # Set up workbook in constant memory mode and enable nan_inf_to_errors to handle float NaNs/Infs without crashing
        workbook = xlsxwriter.Workbook(
            final_excel_path, 
            {
                'constant_memory': True,
                'nan_inf_to_errors': True
            }
        )
        workbook.use_zip64()
        
        for i, filename in enumerate(files):
            # Clean sheet name to preserve county identity (e.g., "Albany_upgrade0")
            county_name = filename.replace('.csv', '')
            file_path = os.path.join(SOURCE_DIR, filename)
            
            print(f"  [{i+1}/{len(files)}] Loading {county_name} into processor RAM")
            
            # cleaning na values and avoiding Excel errors when working with massive datasets
            df = pd.read_csv(file_path, low_memory=False).fillna('')
            headers = df.columns.tolist()
            data = df.values.tolist()
            total_rows = len(data)

            def write_subset_to_sheet(sheet_name, row_subset):
                print(f"    -> Writing Tab: {sheet_name} ({len(row_subset)} rows)...")

                # Adding in the county name column to the data so we can retain the origin in the master sheet
                # Truncating to 31 characters max (Excel tab length limit)
                worksheet = workbook.add_worksheet(sheet_name[:31])
                
                # Writing Headers and row by row data processing
                for col_num, header in enumerate(headers):
                    worksheet.write(0, col_num, header)
                 
                # Compiling the data
                for row_num, row_values in enumerate(row_subset):
                    for col_num, cell_value in enumerate(row_values):
                        worksheet.write(row_num + 1, col_num, cell_value)

            # Handle splitting for massive counties
            if total_rows <= ROW_LIMIT:
                write_subset_to_sheet(county_name, data)
            else:
                num_parts = (total_rows // ROW_LIMIT) + 1
                print(f"    [!] {county_name} exceeds limit. Splitting into {num_parts} parts.")
                for p in range(num_parts):
                    start = p * ROW_LIMIT
                    end = (p + 1) * ROW_LIMIT
                    subset = data[start:end]
                    write_subset_to_sheet(f"{county_name[:25]}_P{p+1}", subset)
                    del subset

            # Clearing the memory so the next round can take place
            del df
            del data
            print(f"  [v] {county_name} Finished.")

        # Finalizing the workbook and saving the file for this scenario
        print(f"\nClosing out the workbook and exporting the final file ")
        workbook.close()
        print(f"SUCCESS! Your Upgrade {upgrade} Master is ready: {final_excel_path}")


if __name__ == "__main__":
    build_full_master_excel()


Processing Scenario: Upgrade 0
--- Starting Master Excel Build with 62 Counties for Upgrade 0 ---
  [1/62] Loading Albany_upgrade0 into processor RAM
    -> Writing Tab: Albany_upgrade0 (11656 rows)...
  [v] Albany_upgrade0 Finished.
  [2/62] Loading Allegany_upgrade0 into processor RAM
    -> Writing Tab: Allegany_upgrade0 (1252 rows)...
  [v] Allegany_upgrade0 Finished.
  [3/62] Loading Bronx_upgrade0 into processor RAM
    -> Writing Tab: Bronx_upgrade0 (14989 rows)...
  [v] Bronx_upgrade0 Finished.
  [4/62] Loading Broome_upgrade0 into processor RAM
    -> Writing Tab: Broome_upgrade0 (6099 rows)...
  [v] Broome_upgrade0 Finished.
  [5/62] Loading Cattaraugus_upgrade0 into processor RAM
    -> Writing Tab: Cattaraugus_upgrade0 (2608 rows)...
  [v] Cattaraugus_upgrade0 Finished.
  [6/62] Loading Cayuga_upgrade0 into processor RAM
    -> Writing Tab: Cayuga_upgrade0 (2504 rows)...
  [v] Cayuga_upgrade0 Finished.
  [7/62] Loading Chautauqua_upgrade0 into processor RAM
    -> Writing 

## (4) Data Sources, Logic, and Baseline Understanding ##

- **California Boundaries**: Sourced from the [California Open Data Portal](https://data.ca.gov/dataset/ca-geographic-boundaries), a subdivision of the Census Bureau's data specific to the county level
- **ComStock Data**: Sourced from the [OEDI](https://data.openei.org/s3_viewer?bucket=oedi-data-lake&prefix=nrel-pds-building-stock%2Fend-use-load-profiles-for-us-building-stock%2F2025%2Fcomstock_amy2018_release_3%2Fmetadata_and_annual_results%2Fby_state_and_county%2Ffull%2Fcsv%2Fstate%3DCA%2F), this is a source that has the data well-structured for download. Required significant stitching to put together into a concise dataset, attributed at the county level and containing sample data of modeled building. Original source for the data is NREL, set to the **2025 iteration**.
    - **ComStock Crosswalking**: The managed by OEDI/NREL is using a different GEOID system for census tracts and county naming than other data sets, requiring crosswalking to align to the geospatial layout components. Direct linkage does not work, do not try without transforming the IDs!

In [ ]:
import os
import sys
import json
import glob
import requests
import pandas as pd
import geopandas as gpd

# (4.1) Full Robust Interactive Map Generator (Enforced Case-Insensitive Joins)
# ==============================================================================
# Configuration section
SOURCE_DIR = '/Users/neil.stein/Documents/CFB/Western Campaigns/NYSERDA_AC2HP/Data Sources/COMStock'
SHAPEFILE_PATH = 'Data Sources/ny_counties.zip'
SHP_COUNTY_COL = 'NAME'
CURRENT_YEAR = 2026

# The target upgrade scenarios we want to generate separate HTML maps for
TARGET_UPGRADES = [0, 1, 2, 4]

# 'forbidden' files to exclude from the compiler so it doesn't try to write the whole state master file as a county
FORBIDDEN_FILES = [
    "NY_Master_COMStock_Upgrade4_FINAL_COMPLETE.xlsx", "NY_Master_COMStock_Upgrade2_FINAL_COMPLETE.xlsx",
    "NY_Master_COMStock_Upgrade1_FINAL_COMPLETE.xlsx", "NY_Master_COMStock_Upgrade0_FINAL_COMPLETE.xlsx", "~$CA_Master_COMStock_FINAL_COMPLETE.xlsx",
    "~$CA_Master_COMStock_FINAL.xlsx", "~$NY_Master_COMStock_Upgrade0_FINAL_COMPLETE.xlsx"
]

# Metrics map configuration
METRIC_MAP = {
    "in.sqft..ft2": "mean", "in.year_built": "mean", 
    "out.params.hdd50f": "mean", "out.params.hdd65f": "mean",
    "out.params.cdd50f": "mean", "out.params.cdd65f": "mean",
    "out.params.cooling_equipment_capacity..tons": "mean",
    "out.params.dx_cooling_capacity_tons..tons": "mean",
    "out.params.dx_heating_capacity_at_rated..kbtu_per_hr": "mean",
    "out.params.heating_equipment..kbtu_per_hr": "mean",
    "out.params.primary_gas_coil_capacity..kbtu_per_hr": "mean",
    "out.params.supplemental_gas_coil_capacity..kbtu_per_hr": "mean",
    "out.params.dx_heating_fraction_electric_defrost": "mean",
    "out.params.dx_heating_fraction_electric_supplemental": "mean",
    "out.params.dx_heating_fraction_supplemental": "mean",
    "out.params.dx_heating_supplemental_capacity..kbtu_per_hr": "mean",
    "out.params.dx_heating_supplemental_capacity_electric..kbtu_per_hr": "mean",
    "out.params.dx_heating_supplemental_capacity_gas..kbtu_per_hr": "mean",
    "out.electricity.total.energy_consumption..kwh": "sum",
    "out.natural_gas.total.energy_consumption..kwh": "sum",
    "out.propane.total.energy_consumption..kwh": "sum",
    "out.nox_emissions.fuel_oil..nox_kg": "sum",
    "out.nox_emissions.natural_gas..nox_kg": "sum",
    "out.nox_emissions.propane..nox_kg": "sum"
}

def get_external_asset(url):
    try: return requests.get(url, timeout=10).text
    except: return ""

def build_grounded_explorer(upgrade):
    target_pattern = os.path.join(SOURCE_DIR, f"*_upgrade{upgrade}.csv")
    target_csvs = sorted([f for f in glob.glob(target_pattern) if os.path.basename(f) not in FORBIDDEN_FILES])
    
    if not target_csvs:
        print(f"ERROR: No source files found matching pattern {target_pattern}")
        sys.exit()

    print(f"  -> Compiling master data directly from {len(target_csvs)} raw county files...")
    
    T_COL = 'in.comstock_building_type'
    G_COL = 'in.comstock_building_type_group'
    TON_COL = 'out.params.cooling_equipment_capacity..tons'
    
    cols_to_load = [T_COL, G_COL] + [col for col in METRIC_MAP.keys()]
    
    county_aggregated_dfs = []
    audit_data = []
    hierarchy_map = {}

    for file_path in target_csvs:
        # Extract name and clean it up
        raw_county_name = os.path.basename(file_path).replace(f'_upgrade{upgrade}.csv', '').strip()
        try:
            raw_f = pd.read_csv(file_path, usecols=lambda c: c in cols_to_load, low_memory=False)
            raw_f[T_COL] = raw_f[T_COL].astype(str).str.strip()
            raw_f[G_COL] = raw_f[G_COL].astype(str).str.strip()

            for _, row in raw_f[[G_COL, T_COL]].drop_duplicates().iterrows():
                if row[G_COL] not in hierarchy_map: 
                    hierarchy_map[row[G_COL]] = set()
                hierarchy_map[row[G_COL]].add(row[T_COL])

            agg_rules = {col: METRIC_MAP[col] for col in METRIC_MAP.keys() if col in raw_f.columns}
            county_agg = raw_f.groupby(T_COL).agg(agg_rules).reset_index()
            county_agg = county_agg.rename(columns={T_COL: 'building_type'})
            county_agg['clean_county'] = raw_county_name
            
            rename_dict = {col: f"{col}.{METRIC_MAP[col]}" for col in agg_rules.keys()}
            county_agg = county_agg.rename(columns=rename_dict)
            county_aggregated_dfs.append(county_agg)

            if TON_COL in raw_f.columns:
                for b_type, group in raw_f.groupby(T_COL):
                    vals = group[TON_COL].dropna()
                    audit_data.append({
                        'clean_county': raw_county_name, 'building_type': b_type,
                        'true_count': len(group),
                        'bin_data': [
                            len(vals[vals <= 5]), 
                            len(vals[(vals > 5) & (vals <= 10)]), 
                            len(vals[(vals > 10) & (vals <= 15)]), 
                            len(vals[(vals > 15) & (vals <= 20)]), 
                            len(vals[vals > 20])
                        ]
                    })
        except Exception as e:
            print(f"      [!] Failed to compile data for {raw_county_name}: {e}")
            continue

    if not county_aggregated_dfs:
        print("ERROR: Failed to aggregate any county datasets.")
        sys.exit()
        
    master_df = pd.concat(county_aggregated_dfs, ignore_index=True)
    master_df['clean_county'] = master_df['clean_county'].astype(str).str.strip()
    master_df['building_type'] = master_df['building_type'].astype(str).str.strip()

    if "in.year_built.mean" in master_df.columns:
        master_df['Mean_Building_Age_Years'] = CURRENT_YEAR - master_df["in.year_built.mean"]

    df = master_df.merge(pd.DataFrame(audit_data), on=['clean_county', 'building_type'], how='left')

    def generate_tooltip(row):
        bins = row['bin_data'] if isinstance(row['bin_data'], list) else [0,0,0,0,0]
        mx = max(bins) if max(bins) > 0 else 1
        viz = "".join(["█" * int((b/mx)*5) + "░" * (5-int((b/mx)*5)) + " " for b in bins])
        return (f"<div style='font-family:monospace; min-width:240px;'>"
                f"<b style='font-family:sans-serif;'>{row['clean_county']}</b><br>Sector: {row['building_type']}<hr>"
                f"TONNAGE DISTRIBUTION (0-20+)<br>[{viz}]<br>---------------------------<br>"
                f"<b>Count: {int(row['true_count']) if pd.notna(row['true_count']) else 0} Units</b></div>")

    df['tooltip_html'] = df.apply(generate_tooltip, axis=1)
    
    final_cols = ['Mean_Building_Age_Years'] if 'Mean_Building_Age_Years' in df.columns else []
    for m, method in METRIC_MAP.items():
        full_name = f"{m}.{method}"
        if full_name == "in.year_built.mean": continue
        if full_name in df.columns: final_cols.append(full_name)
        elif m in df.columns: final_cols.append(m)

    pivot = df.pivot(index='clean_county', columns='building_type', values=final_cols + ['tooltip_html']).reset_index()
    pivot.columns = [f"{col[0]}__JOIN__ {col[1]}" if col[1] else col[0] for col in pivot.columns]
    
    gdf = gpd.read_file(SHAPEFILE_PATH).to_crs(epsg=4326)
    hierarchy_serializable = {k: sorted(list(v)) for k, v in hierarchy_map.items()}
    
    return pivot, gdf, hierarchy_serializable, final_cols

def generate_ui(pivot, gdf, hierarchy, metrics, upgrade):
    output_name = f"NY_COMStock_Energy_Data_Explorer_Upgrade{upgrade}.html"
    
    # ENFORCED CASE-INSENSITIVE JOIN KEY TO PREVENT BLANK MAPS
    gdf['county_join_key'] = gdf[SHP_COUNTY_COL].astype(str).str.upper().str.strip()
    pivot['county_join_key'] = pivot['clean_county'].astype(str).str.upper().str.strip()
    
    merged = gdf.merge(pivot, on='county_join_key', how='left')
    
    # Safeguard JSON serialization from Timestamp metadata structures
    for col in merged.select_dtypes(include=['datetime64', 'datetimetz']).columns:
        merged[col] = merged[col].astype(str)
        
    geojson_data = merged.to_json()
    
    leaflet_js = get_external_asset("https://unpkg.com/leaflet@1.9.4/dist/leaflet.js")
    leaflet_css = get_external_asset("https://unpkg.com/leaflet@1.9.4/dist/leaflet.css")

    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>New York COMStock Energy Explorer - Based on Upgrade {upgrade} Scenario</title>
        <style>{leaflet_css}</style>
        <script>{leaflet_js}</script>
        <style>
            body {{ margin:0; display:flex; font-family:sans-serif; background:#ecf0f1; overflow:hidden; }}
            #sidebar {{ width:380px; height:100vh; background:#2c3e50; color:white; padding:30px; z-index:2000; flex-shrink:0; overflow-y:auto; }}
            #map {{ flex-grow:1; height:100vh; }}
            #header-box {{ position:absolute; top:0; left:380px; width:calc(100% - 380px); height:75px; background:white; z-index:1000; display:flex; align-items:center; justify-content:center; border-bottom:2px solid #ccc; }}
            #right-ui {{ position:absolute; top:95px; right:25px; z-index:1000; background:white; padding:20px; border-radius:8px; width:280px; box-shadow:0 4px 10px rgba(0,0,0,0.1); }}
            select {{ width:100%; padding:12px; margin-top:10px; font-weight:600; cursor:pointer; }}
            .legend {{ background:white; padding:10px; line-height:18px; color:#555; border-radius:5px; box-shadow:0 4px 10px rgba(0,0,0,0.2); font-size:12px; }}
            .legend i {{ width:18px; height:18px; float:left; margin-right:8px; opacity:0.8; }}
        </style>
    </head>
    <body>
        <div id="sidebar">
            <h1 style="color:#e74c3c; border-left:5px solid #e74c3c; padding-left:10px; font-size:1.4em;">Metric Selector</h1>
            <select id="metric-select" onchange="updateMap()">
                {"".join([f'<option value="{m}">{m.replace("out.params.", "").replace(".mean", " (Avg)").replace(".sum", " (Total)").replace("Mean_Building_Age_Years", "Building Age (Years)")}</option>' for m in metrics])}
            </select>
        </div>
        <div id="header-box"><h2 style="color:#2c3e50; margin:0; font-weight:800; text-align:center;">New York COMStock Energy Explorer - Upgrade {upgrade}</h2></div>
        <div id="right-ui">
            <label style="font-size:0.8em; font-weight:bold; color:#7f8c8d;">1. PARENT GROUP</label>
            <select id="group-select" onchange="updateTypes()"></select>
            <label style="font-size:0.8em; font-weight:bold; color:#7f8c8d; margin-top:15px; display:block;">2. BUILDING SECTOR</label>
            <select id="type-select" onchange="updateMap()"></select>
        </div>
        <div id="map"></div>
        <script>
            const geojson = {geojson_data}, hierarchy = {json.dumps(hierarchy)};
            const map = L.map('map', {{ zoomControl:false }}).setView([43.0, -75.5], 7);
            L.tileLayer('https://{{s}}.basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}{{r}}.png').addTo(map);
            let geoLayer, legend;

            function getColor(d, max) {{
                return d > max * 0.8 ? '#800026' : d > max * 0.6 ? '#BD0026' : d > max * 0.4 ? '#E31A1C' : d > max * 0.2 ? '#FC4E2A' : '#FED976';
            }}

            function updateTypes() {{
                const group = document.getElementById('group-select').value;
                document.getElementById('type-select').innerHTML = (hierarchy[group] || []).map(t => `<option value="${{t}}">${{t}}</option>`).join('');
                updateMap();
            }}

            function updateMap() {{
                if (geoLayer) map.removeLayer(geoLayer);
                if (legend) map.removeControl(legend);
                const metric = document.getElementById('metric-select').value, type = document.getElementById('type-select').value;
                const propKey = metric + "__JOIN__ " + type, toolKey = "tooltip_html__JOIN__ " + type;
                let vals = geojson.features.map(f => f.properties[propKey] || 0).filter(v => v > 0);
                let max = Math.max(...vals, 1);

                geoLayer = L.geoJson(geojson, {{
                    style: f => ({{ fillColor: f.properties[propKey] ? getColor(f.properties[propKey], max) : '#fff', weight:1.5, color:#333', fillOpacity:0.8 }}),
                    onEachFeature: (f, l) => l.bindTooltip(f.properties[toolKey] || "No Data", {{ sticky:true }})
                }}).addTo(map);

                legend = L.control({{position: 'bottomleft'}});
                legend.onAdd = function() {{
                    let div = L.DomUtil.create('div', 'legend'), grades = [0, max*0.2, max*0.4, max*0.6, max*0.8];
                    div.innerHTML = '<b>Metric Scale</b><br>';
                    for (let i = 0; i < grades.length; i++) {{
                        div.innerHTML += '<i style="background:' + getColor(grades[i] + 0.1, max) + '"></i> ' + grades[i].toLocaleString() + (grades[i+1] ? '&ndash;' + grades[i+1].toLocaleString() + '<br>' : '+');
                    }}
                    return div;
                }};
                legend.addTo(map);
            }}
            document.getElementById('group-select').innerHTML = Object.keys(hierarchy).map(g => `<option value="${{g}}">${{g}}</option>`).join('');
            updateTypes();
        </script>
    </body>
    </html>
    """
    with open(output_name, "w", encoding="utf-8") as f: f.write(html_content)
    print(f"SUCCESS: Upgrade {upgrade} HTML Explorer Ready: {output_name}")

if __name__ == "__main__":
    for up in TARGET_UPGRADES:
        print(f"\n--- Initiating Build for Upgrade {up} Interactive Map ---")
        p, g, h, m = build_grounded_explorer(upgrade=up)
        generate_ui(p, g, h, m, upgrade=up)


--- Initiating Build for Upgrade 0 Interactive Map ---
  -> Compiling master data directly from 62 raw county files...
SUCCESS: Upgrade 0 HTML Explorer Ready: NY_COMStock_Energy_Data_Explorer_Upgrade0.html

--- Initiating Build for Upgrade 1 Interactive Map ---
  -> Compiling master data directly from 62 raw county files...
SUCCESS: Upgrade 1 HTML Explorer Ready: NY_COMStock_Energy_Data_Explorer_Upgrade1.html

--- Initiating Build for Upgrade 2 Interactive Map ---
  -> Compiling master data directly from 62 raw county files...
SUCCESS: Upgrade 2 HTML Explorer Ready: NY_COMStock_Energy_Data_Explorer_Upgrade2.html

--- Initiating Build for Upgrade 4 Interactive Map ---
  -> Compiling master data directly from 62 raw county files...
SUCCESS: Upgrade 4 HTML Explorer Ready: NY_COMStock_Energy_Data_Explorer_Upgrade4.html


**Personal Commentary:**


The original pass of data analysis yielded a map that is basically just a nice visual tool to compare the entirety of a key metric across different counties, but there is no useful nuance here. Also the HTML was clunky, so the next pass is going to be more focused. Retaining this version though for understanding the evolution in design!

In [ ]:
import os
import sys
import json
import glob
import requests
import pandas as pd
import geopandas as gpd

# (5) New attempt, new parameters, focused on the types of HVAC equipment in use
# Interactive map will focus on unit types, identifying opportunities based on HVAC unit class
# ==============================================================================
# Configuration section - update the paths and parameters as needed
SOURCE_DIR = 'Data Sources/COMStock/' 
SHAPEFILE_PATH = 'Data Sources/ny_counties.zip'
# Confirm that the shapefile county column matches this variable
SHP_COUNTY_COL = 'NAME'

# Target upgrades scenarios to iterate over
TARGET_UPGRADES = [0, 1, 2, 4]

# Target dataset columns, modifiable as needed for this layer of comparison
COL_B_TYPE = 'in.comstock_building_type'
COL_B_GROUP = 'in.comstock_building_type_group'
COL_HVAC = 'in.hvac_combined_type'
COL_SQFT = 'calc.weighted.sqft..ft2'
COL_TONS = 'out.params.dx_cooling_capacity_tons..tons'

# Climate columns that are used for the tooltip & ranking
CLIMATE_COLS = [
    'out.params.cdd50f', 'out.params.cdd65f', 
    'out.params.hdd50f', 'out.params.hdd65f'
]

# Grouping bins for tonnage (change per analysis targets)
BIN_EDGES = [0, 5, 10, 15, 20, 30, 50, 75, 100, 150, 200, float('inf')]
BIN_LABELS = ['0-5', '6-10', '11-15', '16-20', '21-30', '31-50', '51-75', '76-100', '101-150', '151-200', '200+']


# Functions Section - building out the dataset and the interactive map
# ==============================================================================
def building_HVAC_explorer(upgrade):
    target_pattern = os.path.join(SOURCE_DIR, f"*_upgrade{upgrade}.csv")
    target_csvs = [f for f in glob.glob(target_pattern)]
    hvac_data, hierarchy_map = [], {}
    
    REQUIRED_COLS = [COL_B_TYPE, COL_B_GROUP, COL_HVAC, COL_SQFT, COL_TONS] + CLIMATE_COLS

    if not target_csvs:
        print(f"ERROR: No CSV source files found for Upgrade {upgrade} in {SOURCE_DIR}")
        sys.exit()

    print(f"V36 Compiling HVAC Data from {len(target_csvs)} CSV files for Upgrade {upgrade}")

    # Aggregating the data from the source folder, preprocessing the data to get our tonnage results
    for file_path in target_csvs:
        county_name = os.path.basename(file_path).replace(f'_upgrade{upgrade}.csv', '').strip()
        try:
            # Only load the columns we need for optimized RAM usage
            raw = pd.read_csv(file_path, usecols=lambda c: c in REQUIRED_COLS, low_memory=False)
            raw[COL_TONS] = raw[COL_TONS].fillna(0).astype(float)
            raw[COL_B_TYPE] = raw[COL_B_TYPE].astype(str).str.strip()
            raw[COL_B_GROUP] = raw[COL_B_GROUP].astype(str).str.strip()
            raw[COL_HVAC] = raw[COL_HVAC].astype(str).str.strip()
            
            # Bin tagging based on configuration section values
            raw['capacity_bin'] = pd.cut(raw[COL_TONS], bins=BIN_EDGES, labels=BIN_LABELS, include_lowest=True).astype(str)

            # Building out the map hierarchy
            for g, t in raw[[COL_B_GROUP, COL_B_TYPE]].drop_duplicates().values:
                if g not in hierarchy_map: 
                    hierarchy_map[g] = set()
                hierarchy_map[g].add(t)

            # Heart of the aggregation code 
            agg_rules = {
                COL_SQFT: 'sum',
                COL_TONS: 'sum'
            }
            for c in CLIMATE_COLS:
                if c in raw.columns:
                    agg_rules[c] = 'mean'
                    
            main_agg = raw.groupby([COL_B_GROUP, COL_B_TYPE, COL_HVAC], observed=True).agg(agg_rules).reset_index()
            
            # Rename the aggregation outputs to maintain standard structures
            rename_map = {COL_SQFT: 'weighted_sqft', COL_TONS: 'total_tons'}
            for c in CLIMATE_COLS:
                if c in raw.columns:
                    rename_map[c] = f"{c}_avg"
            main_agg = main_agg.rename(columns=rename_map)

            # Bin distribution creation for the tooltip
            bin_agg = raw.groupby([COL_B_TYPE, COL_HVAC, 'capacity_bin'], observed=True)[COL_TONS].sum().unstack(fill_value=0)
            bin_agg_json = bin_agg.apply(lambda x: json.dumps(x.to_dict()), axis=1).reset_index(name='bin_distribution_json')
            
            final_agg = main_agg.merge(bin_agg_json, on=[COL_B_TYPE, COL_HVAC], how='left')
            final_agg['clean_county'] = county_name
            hvac_data.append(final_agg)
        except Exception as e:
            print(f"  [!] Skipping file {county_name} due to processing error: {e}")
            continue

    if not hvac_data:
        print(f"ERROR: Failed to process or load any county CSV files for Upgrade {upgrade}.")
        sys.exit()

    df = pd.concat(hvac_data, ignore_index=True)

    # Share Calculation - Total Tons for the building type in that county to rank the relative importance of findings
    sector_totals = df.groupby(['clean_county', COL_B_TYPE], observed=True)['total_tons'].transform('sum')
    df['share_pct'] = (df['total_tons'] / sector_totals * 100).fillna(0)

    # Geographic Ranking - cross comparing counties against each other
    climate_avg_cols = [f"{c}_avg" for c in CLIMATE_COLS if f"{c}_avg" in df.columns]
    geo_climate = df.groupby(['clean_county', COL_B_TYPE], observed=True)[climate_avg_cols].mean().reset_index()
    
    for c in CLIMATE_COLS:
        avg_col = f"{c}_avg"
        if avg_col in geo_climate.columns:
            rank_name = f"{c.split('.')[-1]}_rank"
            geo_climate[rank_name] = geo_climate.groupby(COL_B_TYPE, observed=True)[avg_col].rank(method='min', ascending=False)
    
    rank_cols = [f"{c.split('.')[-1]}_rank" for c in CLIMATE_COLS if f"{c.split('.')[-1]}_rank" in geo_climate.columns]
    df = df.merge(geo_climate[['clean_county', COL_B_TYPE] + rank_cols + climate_avg_cols], on=['clean_county', COL_B_TYPE], how='left', suffixes=('', '_dup'))
    
    # Drop any duplicated helper average columns created by the merge
    df = df.loc[:, ~df.columns.str.endswith('_dup')]

    # Market Mix Context - understanding what the local market looks like in terms of HVAC distribution to identify strategic opportunities
    hvac_list_df = df.groupby(['clean_county', COL_B_TYPE], observed=True).apply(
        lambda x: json.dumps(dict(zip(x[COL_HVAC], x['total_tons']))),
        include_groups=False
    ).reset_index(name='county_context_json')
    df = df.merge(hvac_list_df, on=['clean_county', COL_B_TYPE], how='left')

    master_csv_output = f"HVAC_Tonnage_Grouping_Master_Final_Upgrade{upgrade}.csv"
    df.to_csv(master_csv_output, index=False)
    print(f"  -> Saved master data to {master_csv_output}")

    # Creating the pivoted data structure for the UI, ensuring all necessary columns are included for the toggles and tooltips
    pivot_cols = {
        'total_tons': 'tons', 
        'share_pct': 'share',
        'bin_distribution_json': 'bins',
        'county_context_json': 'hvac_list'
    }

    for c in CLIMATE_COLS:
        short = c.split('.')[-1]
        if f"{c}_avg" in df.columns:
            pivot_cols[f"{c}_avg"] = f"{short}_val"
        if f"{short}_rank" in df.columns:
            pivot_cols[f"{short}_rank"] = f"{short}_rank"
    
    pivot = df.pivot(index=['clean_county', COL_B_TYPE], columns=COL_HVAC, values=list(pivot_cols.keys()))
    pivot.columns = [f"{pivot_cols[col[0]]}|{col[1]}" for col in pivot.columns]
    pivot = pivot.reset_index().rename(columns={COL_B_TYPE: 'b_type'})

    gdf = gpd.read_file(SHAPEFILE_PATH).to_crs(epsg=4326)
    return pivot, gdf, {k: sorted(list(v)) for k, v in hierarchy_map.items()}, df[COL_HVAC].unique().tolist()

# Function to build the interactive UI, embedding the necessary data and scripts for the toggles and map interactions
def generate_v36_ui(pivot, gdf, hierarchy, hvac_types, upgrade):
    output_html = f"NY_HVAC_Strategic_Explorer_V36_Upgrade{upgrade}.html"
    
    # New York (perhaps other states too) Specific Issue -- there are counties with "ST." or "ST " in their names that need to be standardized for the join
    gdf['county_join_key'] = gdf[SHP_COUNTY_COL].astype(str).str.upper().str.strip().str.replace(r'\bST\.?\s', 'SAINT ', regex=True)
    pivot['county_join_key'] = pivot['clean_county'].astype(str).str.upper().str.strip().str.replace(r'\bST\.?\s', 'SAINT ', regex=True)
    
    merged = gdf.merge(pivot, on='county_join_key', how='left')
    
    # Hard setting the datetime columns to string to avoid issues with GeoJSON
    for col in merged.select_dtypes(include=['datetime64', 'datetimetz']).columns:
        merged[col] = merged[col].astype(str)
        
    geojson_data = merged.to_json()
    print(f"Building HTML Interactive Map containing {len(geojson_data)} characters of GeoJSON data...")

    # HTML Leaflet template with embedded data and scripts for the interactive map
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>New York COMStock HVAC Data Explorer - Based on Data from Upgrade {upgrade}</title> 
        <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
        <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
        <style>
            body {{ margin:0; display:flex; font-family:sans-serif; background:#111; color:#eee; overflow:hidden; }}
            #sidebar {{ width:450px; height:100vh; background:#1a1a1a; padding:20px; border-right:1px solid #333; box-sizing:border-box; overflow-y:auto; z-index:1001; }}
            #map-container {{ flex-grow:1; position:relative; height:100vh; }}
            #map {{ width:100%; height:100%; background:#000; }}
            
            #map-title {{ position:absolute; top:20px; left:50%; transform:translateX(-50%); z-index:1000; background:rgba(0,0,0,0.85); padding:12px 30px; border:1px solid #444; border-radius:4px; text-align:center; min-width:350px; }}
            #map-title h1 {{ margin:0; font-size:1.1em; text-transform:uppercase; color:#ff4444; letter-spacing:1px; }}
            
            .ctrl-box {{ background:#252525; padding:15px; border-radius:8px; margin-bottom:15px; border:1px solid #444; }}
            label {{ font-size:0.65em; font-weight:bold; color:#888; text-transform:uppercase; display:block; margin-bottom:5px; }}
            select {{ width:100%; padding:8px; background:#333; color:white; border:1px solid #555; border-radius:4px; margin-bottom:10px; font-size:0.85em; }}
            
            /* TOGGLE STYLING */
            .toggle-container {{ display:flex; background:#111; border-radius:4px; padding:2px; margin-bottom:10px; border:1px solid #444; }}
            .toggle-btn {{ flex:1; text-align:center; padding:6px; font-size:0.7em; cursor:pointer; color:#888; transition:0.2s; text-transform:uppercase; font-weight:bold; }}
            .toggle-btn.active {{ background:#ff4444; color:white; border-radius:2px; }}

            .grand-total {{ background:#ff4444; padding:15px; border-radius:8px; text-align:center; margin-bottom:20px; }}
            .gt-val {{ font-size:1.8em; font-weight:900; display:block; }}
            .section-header {{ font-size:0.7em; font-weight:bold; color:#ff4444; text-transform:uppercase; margin:25px 0 10px 0; border-bottom:1px solid #333; padding-bottom:5px; }}
            
            .bin-row {{ margin-bottom:8px; }}
            .bin-meta {{ display:flex; justify-content:space-between; font-size:0.7em; color:#bbb; margin-bottom:4px; }}
            .bin-bar-bg {{ background:#000; height:6px; border-radius:3px; width:100%; overflow:hidden; }}
            .bin-bar-fill {{ background:#ff4444; height:100%; }}
            
            .legend {{ background:rgba(0,0,0,0.9); padding:10px; border-radius:5px; color:white; font-size:10px; border:1px solid #444; }}
            .legend i {{ width:14px; height:14px; float:left; margin-right:8px; opacity:0.8; }}
            
            .tt-header {{ font-size:1.1em; font-weight:bold; color:#ff4444; margin-bottom:5px; display:block; }}
            .tt-row {{ display:flex; justify-content:space-between; font-size:0.85em; min-width:210px; margin-top:2px; }}
        </style>
    </head>
    <body>
        <div id="sidebar">
            <div class="grand-total" id="gt-card" style="display:none;">
                <span id="gt-name" style="font-size:0.7em; text-transform:uppercase;">County</span>
                <span class="gt-val" id="gt-val">0</span>
                <span id="gt-label" style="font-size:0.65em; opacity:0.8;">Capacity (Tons)</span>
            </div>
            
            <div class="ctrl-box">
                <label>Select Metric</label>
                <div class="toggle-container">
                    <div id="btn-tons" class="toggle-btn active" onclick="setMetric('tons')">Total Tons</div>
                    <div id="btn-share" class="toggle-btn" onclick="setMetric('share')">Market Share %</div>
                </div>
                
                <label>HVAC System</label>
                <select id="hvac-select" onchange="updateMap()">
                    {"".join([f'<option value="{t}">{t}</option>' for t in hvac_types])}
                </select>
                <label>Building Category</label>
                <select id="group-select" onchange="updateTypes()"></select>
                <label>Specific Building Type</label>
                <select id="type-select" onchange="updateMap()"></select>
            </div>

            <div id="sidebar-content" style="display:none;">
                <div class="section-header">Capacity Distribution (11 Bins)</div>
                <div id="histogram-anchor"></div>
                <div class="section-header">Market Mix for County</div>
                <div id="hvac-list-anchor"></div>
            </div>
        </div>

        <div id="map-container">
            <div id="map-title"><h1 id="title-text">Strategic Explorer - Upgrade {upgrade}</h1></div>
            <div id="map"></div>
        </div>

        <script>
            const geojson = {geojson_data}, hierarchy = {json.dumps(hierarchy)};
            let activeMetric = 'tons';
            const map = L.map('map', {{ zoomControl:false }}).setView([43.0, -75.5], 7); // Centered on NY State
            L.tileLayer('https://{{s}}.basemaps.cartocdn.com/dark_all/{{z}}/{{x}}/{{y}}{{r}}.png').addTo(map);
            let geoLayer, legend;

            function setMetric(m) {{
                activeMetric = m;
                document.getElementById('btn-tons').classList.toggle('active', m === 'tons');
                document.getElementById('btn-share').classList.toggle('active', m === 'share');
                updateMap();
            }}

            function updateTypes() {{
                const g = document.getElementById('group-select').value;
                document.getElementById('type-select').innerHTML = hierarchy[g].map(t => `<option value="${{t}}">${{t}}</option>`).join('');
                updateMap();
            }}

            function updateMap() {{
                if (geoLayer) map.removeLayer(geoLayer);
                if (legend) map.removeControl(legend);
                
                const hvac = document.getElementById('hvac-select').value;
                const sector = document.getElementById('type-select').value;
                const key = activeMetric + "|" + hvac;
                document.getElementById('title-text').innerText = sector + " (Upgrade {upgrade})";

                const features = geojson.features.filter(f => f.properties.b_type === sector);
                const vals = features.map(f => f.properties[key] || 0);
                const max = Math.max(...vals, 1);

                geoLayer = L.geoJson({{type: "FeatureCollection", features: features}}, {{
                    style: f => ({{ fillColor: getColor(f.properties[key] || 0, max), weight:0.5, color:'#444', fillOpacity:0.75 }}),
                    onEachFeature: (f, l) => {{
                        const p = f.properties;
                        const tons = (p['tons|'+hvac]||0).toLocaleString();
                        const share = (p['share|'+hvac]||0).toFixed(1);
                        
                        const c50 = Math.round(p['cdd50f_val|'+hvac] || 0).toLocaleString();
                        const c50r = p['cdd50f_rank|'+hvac] || '-';
                        const c65 = Math.round(p['cdd65f_val|'+hvac] || 0).toLocaleString();
                        const c65r = p['cdd65f_rank|'+hvac] || '-';
                        const h50 = Math.round(p['hdd50f_val|'+hvac] || 0).toLocaleString();
                        const h50r = p['hdd50f_rank|'+hvac] || '-';
                        const h65 = Math.round(p['hdd65f_val|'+hvac] || 0).toLocaleString();
                        const h65r = p['hdd65f_rank|'+hvac] || '-';

                        const tooltipContent = `
                            <span class="tt-header">${{p.clean_county}}</span>
                            <div class="tt-row"><span>Total Tons:</span><b>${{tons}}</b></div>
                            <div class="tt-row"><span>Market Share:</span><b>${{share}}%</b></div>
                            <div style="height:1px; background:#444; margin:5px 0;"></div>
                            <div class="tt-row"><span>CDD 50:</span><span>${{c50}} (#${{c50r}})</span></div>
                            <div class="tt-row"><span>CDD 65:</span><span>${{c65}} (#${{c65r}})</span></div>
                            <div style="height:1px; background:#333; margin:3px 0;"></div>
                            <div class="tt-row"><span>HDD 50:</span><span>${{h50}} (#${{h50r}})</span></div>
                            <div class="tt-row"><span>HDD 65:</span><span>${{h65}} (#${{h65r}})</span></div>
                        `;
                        
                        l.bindTooltip(tooltipContent, {{ sticky:true, direction:'top', offset:[0,-10] }});
                        l.on('click', () => showSidebar(p, hvac));
                    }}
                }}).addTo(map);

                legend = L.control({{position: 'bottomright'}});
                legend.onAdd = () => {{
                    let div = L.DomUtil.create('div', 'legend'), gs = [0, max*0.2, max*0.4, max*0.6, max*0.8];
                    div.innerHTML = `<b>${{activeMetric === 'tons' ? 'Total Tons' : 'Share %'}}</b><br>`;
                    gs.forEach(g => {{ div.innerHTML += `<i style="background:${{getColor(g+0.01, max)}}"></i> ${{Math.round(g).toLocaleString()}}${{activeMetric==='share'?'%':''}}+<br>`; }});
                    return div;
                }};
                legend.addTo(map);
            }}

            function getColor(d, max) {{
                if (!d || d <= 0) return '#1a1a1a';
                return d > max*0.8 ? '#800026' : d > max*0.6 ? '#BD0026' : d > max*0.4 ? '#E31A1C' : d > max*0.2 ? '#FC4E2A' : '#FED976';
            }}

            function showSidebar(p, hvac) {{
                document.getElementById('gt-card').style.display = 'block';
                document.getElementById('sidebar-content').style.display = 'block';
                document.getElementById('gt-name').innerText = p.clean_county;
                
                const val = p[activeMetric+'|'+hvac] || 0;
                document.getElementById('gt-val').innerText = activeMetric === 'tons' ? val.toLocaleString() : val.toFixed(1) + '%';
                document.getElementById('gt-label').innerText = activeMetric === 'tons' ? 'Capacity (Tons)' : 'Market Share %';

                // Bins (in Tons)
                const rawBins = p['bins|'+hvac];
                const bins = rawBins ? (typeof rawBins === 'string' ? JSON.parse(rawBins) : rawBins) : {{}};
                const labels = {json.dumps(BIN_LABELS)};
                const maxBin = Math.max(...Object.values(bins), 1);
                let hb = "";
                labels.forEach(lab => {{
                    const v = bins[lab] || 0;
                    hb += `<div class="bin-row">
                        <div class="bin-meta"><span>${{lab}} Tons</span><span>${{v.toLocaleString()}}</span></div>
                        <div class="bin-bar-bg"><div class="bin-bar-fill" style="width:${{(v/maxBin)*100}}%"></div></div>
                    </div>`;
                }});
                document.getElementById('histogram-anchor').innerHTML = hb;

                // Market Mix (Dynamic Metric)
                const rawList = p['hvac_list|'+hvac];
                const list = rawList ? (typeof rawList === 'string' ? JSON.parse(rawList) : rawList) : {{}};
                const totalTons = Object.values(list).reduce((a, b) => a + b, 0);
                const maxH = Math.max(...Object.values(list), 1);
                let hl = "";
                Object.entries(list).sort((a,b) => b[1]-a[1]).forEach(([name, v]) => {{
                    const isSel = name === hvac;
                    const displayVal = activeMetric === 'tons' ? v.toLocaleString() : ((v/totalTons)*100).toFixed(1) + '%';
                    const barWidth = (v/maxH)*100;
                    hl += `<div class="bin-row" style="opacity:${{isSel ? 1 : 0.4}}">
                        <div class="bin-meta"><span>${{name}}</span><span>${{displayVal}}</span></div>
                        <div class="bin-bar-bg"><div class="bin-bar-fill" style="width:${{barWidth}}%; background:${{isSel?'#ff4444':'#555'}}"></div></div>
                    </div>`;
                }});
                document.getElementById('hvac-list-anchor').innerHTML = hl;
            }}

            document.getElementById('group-select').innerHTML = Object.keys(hierarchy).map(g => `<option value="${{g}}">${{g}}</option>`).join('');
            updateTypes();
        </script>
    </body>
    </html>
    """
    with open(output_html, "w", encoding="utf-8") as f: f.write(html_content)
    print(f"SUCCESS: HTML Map Generated for Upgrade {upgrade}: {output_html}")

if __name__ == "__main__":
    for up in TARGET_UPGRADES:
        print(f"\n--- Initiating Build for Upgrade {up} Interactive Map ---")
        p, g, h, t = building_HVAC_explorer(upgrade=up)
        generate_v36_ui(p, g, h, t, upgrade=up)


--- Initiating Build for Upgrade 0 Interactive Map ---
V36 Compiling HVAC Data from 62 CSV files for Upgrade 0
  -> Saved master data to HVAC_Tonnage_Grouping_Master_Final_Upgrade0.csv
Building HTML Interactive Map containing 215155364 characters of GeoJSON data...
SUCCESS: HTML Map Generated for Upgrade 0: NY_HVAC_Strategic_Explorer_V36_Upgrade0.html

--- Initiating Build for Upgrade 1 Interactive Map ---
V36 Compiling HVAC Data from 62 CSV files for Upgrade 1
  -> Saved master data to HVAC_Tonnage_Grouping_Master_Final_Upgrade1.csv
Building HTML Interactive Map containing 215155385 characters of GeoJSON data...
SUCCESS: HTML Map Generated for Upgrade 1: NY_HVAC_Strategic_Explorer_V36_Upgrade1.html

--- Initiating Build for Upgrade 2 Interactive Map ---
V36 Compiling HVAC Data from 62 CSV files for Upgrade 2
  -> Saved master data to HVAC_Tonnage_Grouping_Master_Final_Upgrade2.csv
Building HTML Interactive Map containing 215155421 characters of GeoJSON data...
SUCCESS: HTML Map Gener

## FINAL VERSION!! ##

In [ ]:
# (6) Using the census tract data in the NLR data set to model the estimated impacts of transition down to the most localizable level
# ==================================================================================================
# VERSION 113
# Configuration section - update columns, names, labels as needed but a lot of this data is interdependent so be careful!
VERSION = "NY_Final_Census_Tract_ANALYSIS_V1.1"

# Data extraction columns that are useful for our analysis, some can be pared down but there are dependencies at play here
# Specifically, "'in.nhgis_tract_gisjoin','in.comstock_building_type', 'in.cec_climate_zone', 'in.year_built','in.sqft..ft2', 'weight','out.params.cooling_equipment_capacity..tons'"
# are essential or the code will collapse
COLUMNS_TO_KEEP = [
    'in.nhgis_tract_gisjoin','in.comstock_building_type', 'in.cec_climate_zone', 'in.sqft..ft2', 'weight', 
    'in.heating_fuel', 'in.year_built', 'out.params.hdd50f', 'out.params.hdd65f', 'in.hvac_vent_type',
    'out.params.cdd50f', 'out.params.cdd65f', 'out.params.cooling_equipment_capacity..tons', 
    'out.params.dx_cooling_capacity_tons..tons', 'out.params.dx_heating_capacity_at_rated..kbtu_per_hr', 
    'out.params.heating_equipment..kbtu_per_hr', 'out.params.primary_gas_coil_capacity..kbtu_per_hr', 
    'out.params.supplemental_gas_coil_capacity..kbtu_per_hr', 'out.params.dx_heating_fraction_electric_defrost', 
    'out.params.dx_heating_fraction_electric_supplemental', 'out.params.dx_heating_fraction_supplemental', 
    'out.params.dx_heating_supplemental_capacity..kbtu_per_hr', 'out.params.dx_heating_supplemental_capacity_electric..kbtu_per_hr', 
    'out.params.dx_heating_supplemental_capacity_gas..kbtu_per_hr', 'out.electricity.total.energy_consumption..kwh', 
    'out.natural_gas.total.energy_consumption..kwh', 'out.propane.total.energy_consumption..kwh', 
    'out.nox_emissions.fuel_oil..nox_kg', 'out.nox_emissions.natural_gas..nox_kg', 'out.nox_emissions.propane..nox_kg'
]

# Key data sources - update file path names as needed (or to a new state if changing the analysis)
SOURCE_DIR = 'Data Sources/COMStock' # Assuming the prior extraction block (block #2 in this notebook) has occured
LOOKUP_FILE = "Data Sources/NonRes AC2HP_re and so conversion split.xlsx" # EXTERNAL LOOKUP FILE for the NonRes AC to HP conversion and RE/SO split
# REPLACE THIS FILE WITH THE CORRECT COLUMNS FROM THE DATASET IF YOU PREFER!!   
TRACT_SHP = 'Data Sources/NY_tl_2022_36_tract.zip' # Census tract shapefile for NY State

# California geospatial files for direct matching or fallbacks
# Remove this if needed, but the CEC climate zone shapefile is useful for confirming that the climate zones are being assigned correctly
# You will need to trim the code below to skip the climate zone assignment if you wish to rely on the 'in.ashrae_iecc_climate_zone_2006'
CEC_CLIMATE_ZONE_SHP_ZIP = "Data Sources/BuildingClimateZones_CEC_2015_8690419816209774084.zip"
UTILITY_SHP_ZIP = "Data Sources/NYS Electric Utility Service Territories_20260710.zip"

# Output file names section, change as needed (helpful for confirming that big data processing is working correctly)
INTERMEDIARY_CSV = "Master_Inventory_Intermediary_Verification.csv"
OUTPUT_CSV = f"Tract_Level_COMStock_Full_Statewide_NY_{VERSION}.csv"
OUTPUT_HTML = f"Census_Tract_COMStock_Map_NY_{VERSION}.html"

# Change the IOUs as needed, these are the ones present in the NonRes AC2HP lookup file for RE and SO conversion that we had good data for
# THIS IS CALIFORNIA SPECIFIC
TARGET_IOUS = ["PG&E", "SDG&E", "SCE", "SMUD", "CPAU"]
# THIS SQUARE FOOTAGE IS ALSO CALIFORNIA SPECIFIC, but it is used to calculate the weighted average of the RE and SO split for the NonRes AC to HP conversion
SQFT_SO, SQFT_RE = 5502, 24563
MAP_BINS = [0, 100, 500, 1000] 

# Target lookup column labels (to change from the technical jargon column ID)
COL_KWH = "Annual Elec Savings (kWh)"
COL_GHG = "Annual GHG savings (tons)"
COL_NPV = "NPV (On-bill)"
COL_THERMS = "Annual Gas Savings (therms)"

# Dictionary to convert coded section in the GEOID to useful names for visual confirmation (updated for you to NY State)
COUNTY_TO_FIPS = {
    "Albany": "36001", "Allegany": "36003", "Bronx": "36005", "Broome": "36007", "Cattaraugus": "36009",
    "Cayuga": "36011", "Chautauqua": "36013", "Chemung": "36015", "Chenango": "36017", "Clinton": "36019",
    "Columbia": "36021", "Cortland": "36023", "Delaware": "36025", "Dutchess": "36027", "Erie": "36029",
    "Essex": "36031", "Franklin": "36033", "Fulton": "36035", "Genesee": "36037", "Greene": "36039",
    "Hamilton": "36041", "Herkimer": "36043", "Jefferson": "36045", "Kings": "36047", "Lewis": "36049",
    "Livingston": "36051", "Madison": "36053", "Monroe": "36055", "Montgomery": "36057", "Nassau": "36059",
    "New York": "36061", "Niagara": "36063", "Oneida": "36065", "Onondaga": "36067", "Ontario": "36069",
    "Orange": "36071", "Orleans": "36073", "Oswego": "36075", "Otsego": "36077", "Putnam": "36079",
    "Queens": "36081", "Rensselaer": "36083", "Richmond": "36085", "Rockland": "36087", "St. Lawrence": "36089",
    "Saratoga": "36091", "Schenectady": "36093", "Schoharie": "36095", "Schuyler": "36097", "Seneca": "36099",
    "Steuben": "36101", "Suffolk": "36103", "Sullivan": "36105", "Tioga": "36107", "Tompkins": "36109",
    "Ulster": "36111", "Warren": "36113", "Washington": "36115", "Wayne": "36117", "Westchester": "36119",
    "Wyoming": "36121", "Yates": "36123"
}

FIPS_TO_COUNTY = {v: k for k, v in COUNTY_TO_FIPS.items()}

# Data cleaning for the NPV finance values to run effectively
def clean_currency(v):
    if pd.isna(v) or v == '': return 0.0
    return float(str(v).replace('$', '').replace(',', '').replace('(', '-').replace(')', '').strip())

def clean_cz_label(val):
    s = str(val).strip().lower().replace('cz', '')
    if '.' in s:
        s = s.split('.')[0]
    return f"cz{s.zfill(2)}"

# Aligning the building year to the vintages present in the TRC study referred to in the NonRes AC to HP Spreadsheet
# Building vintages can also be aligned to coding standard years by state
# (Modify these if new data has better breakdowns)
def map_vintage_bin(year):
    try:
        yr = int(year)
        if yr <= 1979: return 'v1'
        if 1980 <= yr <= 2003: return 'v2'
        return 'v3'
    except: return 'v1'

def extract_numeric_digits(val):
    s = str(val).strip()
    return "".join([c for c in s if c.isdigit()])

def clean_sub_tract_label(val):
    s = str(val).strip()
    if '.' in s:
        return s.split('.')[0]
    return s

# Geospatial mapping with fallbacks to correct the disconnect between the ComStock GEOIDs and standard modern Census Tracts
# We flagged in the initial analysis that the ComStock GEOIDs were not always aligned with the modern Census Tract GEOID (using the 2010 Census Tract GEOIDs)
# so we need to implement a fallback matching system
def match_geoid_with_suffix_fallback(raw_tract_string, target_tract_pool, nhgis_suffix, global_pool):
    if len(raw_tract_string) >= 12 and raw_tract_string.startswith('G'):
        matched_geoid_target = raw_tract_string[1:12]
    else:
        matched_geoid_target = None

    if matched_geoid_target and matched_geoid_target in target_tract_pool:
        return matched_geoid_target

    if matched_geoid_target and len(matched_geoid_target) >= 2:
        truncated_geoid = matched_geoid_target[:-2]
        for geoid_candidate in target_tract_pool:
            if geoid_candidate.endswith(truncated_geoid) or truncated_geoid.endswith(geoid_candidate[-len(truncated_geoid):]):
                return geoid_candidate

    for geoid_candidate in target_tract_pool:
        if nhgis_suffix and geoid_candidate[-6:] == nhgis_suffix:
            return geoid_candidate

    if nhgis_suffix and len(nhgis_suffix) >= 4:
        parent_suffix_prefix = nhgis_suffix[:4]  
        for geoid_candidate in target_tract_pool:
            if geoid_candidate[-6:].startswith(parent_suffix_prefix):
                return geoid_candidate

    if matched_geoid_target and matched_geoid_target in global_pool:
        return matched_geoid_target

    for geoid_candidate in global_pool:
        if nhgis_suffix and geoid_candidate[-6:] == nhgis_suffix:
            return geoid_candidate
            
    return None

# Core code block
def run_pipeline():
    if os.path.exists(INTERMEDIARY_CSV):
        os.remove(INTERMEDIARY_CSV)

    print("\n--- [TRACE] LOADING LOOKUPS & GEOSPATIAL LAYERS ---")
    
    raw_lookup = []
    for sheet_name, sector in [('re_conversion', 're'), ('so_conversion', 'so')]:
        sheet = pd.read_excel(LOOKUP_FILE, sheet_name=sheet_name)
        for _, r in sheet.iterrows():
            pkg = str(r['Package']).lower()
            if '-hp-' not in pkg: continue
            v_bin = 'v1' if '-v1' in pkg else ('v2' if '-v2' in pkg else 'v3')
            u_raw = str(r['IOU territory']).upper()
            iou = 'PG&E' if 'PACIFIC GAS' in u_raw or u_raw == 'PGE' else \
                  ('SCE' if any(x in u_raw for x in ['EDISON', 'SCG', 'SCE']) else \
                  ('SDG&E' if 'SAN DIEGO' in u_raw else u_raw.strip()))
            cz = clean_cz_label(r['CZ'])
            
            therms_val = float(str(r[COL_THERMS]).replace(',', '').strip()) if COL_THERMS in sheet.columns and not pd.isna(r[COL_THERMS]) else 0.0
            
            raw_lookup.append({
                'sector': sector, 'v_bin': v_bin, 'iou': iou, 'cz': cz, 
                'kwh': r[COL_KWH], 'ghg': r[COL_GHG], 'npv': clean_currency(r[COL_NPV]), 'therms': therms_val
            })
    
    lookup_df = pd.DataFrame(raw_lookup)
    target_df = lookup_df[lookup_df['iou'].isin(TARGET_IOUS)]
    # cz_averages was calculated to be used as a fallback for missing data/bad GEOID matches
    cz_averages = target_df.groupby(['sector', 'v_bin', 'cz'])[['kwh', 'ghg', 'npv', 'therms']].mean().to_dict('index')
    direct_lookup = lookup_df.groupby(['sector', 'v_bin', 'iou', 'cz'])[['kwh', 'ghg', 'npv', 'therms']].mean().to_dict('index')
    print("Excel Lookups loaded")

    # Load Census Tract Spatial Geometry Layer
    raw_tracts = gpd.read_file(TRACT_SHP).to_crs("EPSG:4269")
    raw_tracts['GEOID_STR'] = raw_tracts['GEOID'].astype(str).str.strip()
    raw_tracts['GEOID_BASE'] = raw_tracts['GEOID_STR'].apply(lambda x: x[-11:] if len(x) >= 11 else x.zfill(11))
    raw_tracts['NAME_CLEAN'] = raw_tracts['NAME'].apply(clean_sub_tract_label)
    raw_tracts['NAMELSAD_CLEAN'] = raw_tracts['NAMELSAD'].apply(lambda x: "Census Tract " + clean_sub_tract_label(str(x).replace("Census Tract ", "")))
    
    agg_rules = {}
    for col in raw_tracts.columns:
        if col in ['geometry', 'GEOID_BASE']: continue
        if col in ['ALAND', 'AWATER']: agg_rules[col] = 'sum'
        else: agg_rules[col] = 'first'
            
    tracts_gdf = raw_tracts.dissolve(by='GEOID_BASE', aggfunc=agg_rules).reset_index()
    all_california_tracts = set(tracts_gdf['GEOID_BASE'].unique())

    # Setting the CRS directly to avoid potential issues
    # Be sure to check your CRS and adjust if necessary for your specific data, some visual tools like Leaflet use different ones!
    print("--- [TRACE] GEOSPATIAL JOIN FOR OFFICIAL UTILITIES AND CLIMATE ZONES ---")
    cz_gdf = gpd.read_file(CEC_CLIMATE_ZONE_SHP_ZIP).to_crs("EPSG:4269")
    util_gdf = gpd.read_file(UTILITY_SHP_ZIP).to_crs("EPSG:4269")
    
    # Identify Climate Zone column dynamically in case the labeling is inconsistent
    cz_col = None
    for col in ['Zone', 'ClimateZone', 'BZone', 'BUILDING_C', 'CZ', 'BA_Zone']:
        if col in cz_gdf.columns:
            cz_col = col
            break
    if not cz_col:
        zone_cols = [c for c in cz_gdf.columns if 'zone' in c.lower()]
        cz_col = zone_cols[0] if zone_cols else cz_gdf.columns[0]

    # Fixing the names to make mapping the utilities easier
    # (this is hardcode please update if in a different state or if the utility names change)
    def identify_core_iou(row):
        for col in util_gdf.columns:
            val = str(row[col]).upper()
            if 'PACIFIC GAS' in val or 'PGE' in val or 'PG&E' in val: return 'PG&E'
            if 'EDISON' in val or 'SCE' in val: return 'SCE'
            if 'SAN DIEGO' in val or 'SDG&E' in val or 'SDGE' in val: return 'SDG&E'
            if 'SMUD' in val or 'SACRAMENTO MUNICIPAL' in val: return 'SMUD'
            if 'CPAU' in val or 'PALO ALTO' in val: return 'CPAU'
        return None

    util_gdf['Core_IOU'] = util_gdf.apply(identify_core_iou, axis=1)
    
    # Filter to create target anchor layer for nearest neighbor assignment
    # (if IOU is not present or if the census tract is not in an IOU zone)
    core_util_gdf = util_gdf[util_gdf['Core_IOU'].notna()].copy()
    if len(core_util_gdf) == 0:
        core_util_gdf = util_gdf.copy()
        # Backup assignment if no IOU is found, this is a fallback and should be updated if the utility names change
        core_util_gdf['Core_IOU'] = 'PG&E'

    # Mapping using point-based approach in case there are CZ discrepancies or Utility gaps
    tracts_rep = tracts_gdf.copy()
    tracts_rep['geometry'] = tracts_rep.geometry.representative_point()
    
    cz_join = gpd.sjoin_nearest(tracts_rep[['geometry']], cz_gdf[[cz_col, 'geometry']], how='left')
    cz_join = cz_join[~cz_join.index.duplicated(keep='first')]
    
    util_join = gpd.sjoin_nearest(tracts_rep[['geometry']], core_util_gdf[['Core_IOU', 'geometry']], how='left')
    util_join = util_join[~util_join.index.duplicated(keep='first')]

    tract_meta = {}
    geo_fence_registry = {} 
    
    # Cleaning the names to assist the matching
    for idx, r in tracts_gdf.iterrows():
        gid = r['GEOID_BASE']
        state_fp = str(r['STATEFP']).strip().zfill(2)
        county_fp = str(r['COUNTYFP']).strip().zfill(3) 
        full_county_fips = state_fp + county_fp
        
        if county_fp not in geo_fence_registry:
            geo_fence_registry[county_fp] = []
        geo_fence_registry[county_fp].append(gid)
        
        # Backups in case the earlier proximity matching process fails (hardcoded, update if needed)
        acronym = util_join.loc[idx, 'Core_IOU'] if idx in util_join.index else "PG&E"
        raw_cz = cz_join.loc[idx, cz_col] if idx in cz_join.index else "3"
        cz_str = clean_cz_label(raw_cz)
            
        tract_meta[gid] = {
            'u': acronym, 
            'cz': cz_str, 
            'name': r['NAMELSAD_CLEAN'], 
            'co_name': FIPS_TO_COUNTY.get(full_county_fips, "CA")
        }
    print("Spatial geometry adjusted & CZ/Utility Matching Complete")

    # Creating our matching process data base to recieve information from the analysis
    results = {g: {'re_kwh':0.0, 're_ghg':0.0, 're_npv':0.0, 're_tons':0.0, 're_therms':0.0, 'so_kwh':0.0, 'so_ghg':0.0, 'so_npv':0.0, 'so_tons':0.0, 'so_therms':0.0} for g in tracts_gdf['GEOID_BASE']}
    csv_files = sorted([f for f in os.listdir(SOURCE_DIR) if f.endswith('.csv') and 'VERIFY' not in f])
    
    # Tracking the success/failure rate for matches
    match_count = 0
    catchall_count = 0
    is_first_verification_append = True

    print(f"\nProcessing on a County By County Level...")

    # Core function to pull our correct data
    for current_file in csv_files:
        county_name = current_file.replace('.csv', '').strip()
        fips = COUNTY_TO_FIPS.get(county_name)
        if not fips: continue
        
        file_county_3digit = fips[-3:] 
        target_tract_pool = geo_fence_registry.get(file_county_3digit, [])
        fallback_anchor_tract = target_tract_pool[0] if len(target_tract_pool) > 0 else sorted(list(all_california_tracts))[0]
        
        df_county = pd.read_csv(os.path.join(SOURCE_DIR, current_file), low_memory=False)
        
        # Masking to ensure we are looking at our preferred structures
        # This is a hardcode filter for specific building types, fuel types, and HVAC types, update as needed for your analysis
        mask = (df_county['in.comstock_building_type'].str.contains('Retail|Office', na=False, case=False)) & \
               (df_county['in.heating_fuel'] == 'NaturalGas') & (df_county['in.hvac_combined_type'] == 'Central Single-zone RTU_Furnace_DX')
        matched_chunk = df_county[mask][COLUMNS_TO_KEEP].copy()
        matched_chunk['source_county_fips'] = fips
        matched_chunk['v_bin'] = matched_chunk['in.year_built'].apply(map_vintage_bin)
        
        print(f" PROCESSING DATA FILE: {current_file} | Locked Target Pool: {file_county_3digit} ({county_name})")

        if len(matched_chunk) == 0:
            continue

        if is_first_verification_append:
            matched_chunk.to_csv(INTERMEDIARY_CSV, index=False, mode='w')
            is_first_verification_append = False
        else:
            matched_chunk.to_csv(INTERMEDIARY_CSV, index=False, mode='a', header=False)

        local_match = 0
        local_catchall = 0

        # Processing through the correctly matched rows to extract the target data to be deposited in the csv
        # The building type filter (re/so) is hardcoded here, update as needed for your analysis
        for idx, row in matched_chunk.iterrows():
            sector = 're' if 'Retail' in str(row['in.comstock_building_type']) else 'so'
            tons = float(row['out.params.cooling_equipment_capacity..tons']) * row['weight']
            sqft_ratio = (row['in.sqft..ft2'] / (SQFT_RE if sector == 're' else SQFT_SO)) * row['weight']
            
            raw_tract_string = str(row['in.nhgis_tract_gisjoin']).strip()
            tract_numeric = extract_numeric_digits(raw_tract_string)
            nhgis_suffix = tract_numeric[-6:].zfill(6) if tract_numeric else ""
            
            matched_geoid = match_geoid_with_suffix_fallback(raw_tract_string, target_tract_pool, nhgis_suffix, all_california_tracts)

            if not matched_geoid or matched_geoid not in results:
                matched_geoid = fallback_anchor_tract
                local_catchall += 1
                catchall_count += 1
            else:
                local_match += 1
                match_count += 1

            m = tract_meta[matched_geoid]
            results[matched_geoid][f'{sector}_tons'] += tons
            
            data = direct_lookup.get((sector, row['v_bin'], m['u'], m['cz']))
            if not data: data = cz_averages.get((sector, row['v_bin'], m['cz']))
            
            # the building sqft ratio is used to scale the energy and GHG savings to the actual building size
            # recommended to keep some version of this if there is an external model for conversions being used 
            if data:
                results[matched_geoid][f'{sector}_kwh'] += (data['kwh'] * sqft_ratio)
                results[matched_geoid][f'{sector}_ghg'] += (data['ghg'] * sqft_ratio)
                results[matched_geoid][f'{sector}_npv'] += (data['npv'] * sqft_ratio)
                results[matched_geoid][f'{sector}_therms'] += (data['therms'] * sqft_ratio)

        print(f"   -> FILE TOTALS: Matches: {local_match:,} | Fallbacks Recorded: {local_catchall:,}")

    print(f"\nPIPELINE RUN COMPLETE: Total Matches: {match_count:,} | Total Back-Ups Assigned: {catchall_count:,}")

    # Summing and finalizing the data analysis
    print("\n Finalizing the data components")
    res_df = pd.DataFrame.from_dict(results, orient='index').reset_index().rename(columns={'index':'GEOID_BASE'})
    final_gdf = tracts_gdf.merge(res_df, on='GEOID_BASE')
    final_gdf['CEC_Climate_Zone'] = final_gdf['GEOID_BASE'].map(lambda x: tract_meta[x]['cz'])
    final_gdf['Acronym'] = final_gdf['GEOID_BASE'].map(lambda x: tract_meta[x]['u'])
    final_gdf['County_Name'] = final_gdf['GEOID_BASE'].map(lambda x: tract_meta[x]['co_name'])
    final_gdf['total_tons'] = final_gdf['re_tons'] + final_gdf['so_tons']
    
    cols_to_sum = ['re_tons', 're_npv', 're_kwh', 're_ghg', 're_therms', 'so_tons', 'so_npv', 'so_kwh', 'so_ghg', 'so_therms']
    c_sum = final_gdf.groupby(final_gdf['GEOID_BASE'].str[:5])[cols_to_sum].transform('sum')
    final_gdf['co_tons'] = c_sum['re_tons'] + c_sum['so_tons']
    final_gdf['co_npv'] = c_sum['re_npv'] + c_sum['so_npv']
    final_gdf['co_kwh'] = c_sum['re_kwh'] + c_sum['so_kwh']
    final_gdf['co_ghg'] = c_sum['re_ghg'] + c_sum['so_ghg']
    final_gdf['co_therms'] = c_sum['re_therms'] + c_sum['so_therms']
    
    final_gdf.drop(columns='geometry').to_csv(OUTPUT_CSV, index=False)
    geojson = final_gdf.to_crs("EPSG:4326").to_json()

    # HTML Script Initiatiation
    html = f"""
    <!DOCTYPE html><html><head>
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
    <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
    <style>#map {{ height: 100vh; background: #f4f4f4; }}</style>
    </head><body><div id="map"></div><script>
        var map = L.map('map').setView([37.0, -119.5], 6);
        L.tileLayer('https://{{s}}.basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}.png').addTo(map);
        function getColor(v) {{
            var b = {MAP_BINS};
            return v > b[3] ? '#00441b' : v > b[2] ? '#238b45' : v > b[1] ? '#74c476' : '#e5f5e0';
        }}
        var tractLayer = L.geoJson({geojson}, {{
            style: function(f) {{ return {{fillColor: getColor(f.properties.total_tons), weight: 0.3, color: 'white', fillOpacity: 0.85}}; }},
            onEachFeature: function(f, l) {{
                var p = f.properties;
                l.bindTooltip(`
                    <div style="font-size:12px; min-width:260px;">
                        <strong style="font-size:14px;">${{p.County_Name}} | ${{p.NAMELSAD_CLEAN}}</strong><br>
                        <b>CZ:</b> ${{p.CEC_Climate_Zone}} | <b>IOU:</b> ${{p.Acronym}}<br><hr>
                        <table style="width:100%; border-collapse: collapse;">
                            <tr style="border-bottom: 1px solid #ccc"><td><b>Metric</b></td><td><b>Retail</b></td><td><b>Office</b></td></tr>
                            <tr><td>HVAC Tonnage:</td><td>${{Math.round(p.re_tons).toLocaleString()}}</td><td>${{Math.round(p.so_tons).toLocaleString()}}</td></tr>
                            <tr><td>kWh Delta:</td><td>${{Math.round(p.re_kwh).toLocaleString()}}</td><td>${{Math.round(p.so_kwh).toLocaleString()}}</td></tr>
                            <tr><td>Therms Saved:</td><td>${{Math.round(p.re_therms).toLocaleString()}}</td><td>${{Math.round(p.so_therms).toLocaleString()}}</td></tr>
                            <tr><td>GHG Avoided (Tons):</td><td>${{Math.round(p.re_ghg).toLocaleString()}}</td><td>${{Math.round(p.so_ghg).toLocaleString()}}</td></tr>
                            <tr><td>NPV $:</td><td>$${{Math.round(p.re_npv).toLocaleString()}}</td><td>$${{Math.round(p.so_npv).toLocaleString()}}</td></tr>
                        </table><hr>
                        <b>COUNTY CUMULATIVE:</b><br>
                        - ${{Math.round(p.co_tons).toLocaleString()}} Total HVAC Capacity (tons)<br>
                        - ${{Math.round(p.co_kwh).toLocaleString()}} Total kWh Delta<br>
                        - ${{Math.round(p.co_therms).toLocaleString()}} Total Therms Saved<br>
                        - ${{Math.round(p.co_ghg).toLocaleString()}} Total Avoided GHG Emissions (tons)<br>
                        - $${{Math.round(p.co_npv).toLocaleString()}} Total NPV $ Saved
                    </div>`, {{sticky: true}});
            }}
        }}).addTo(map);
    </script></body></html>
    """
    with open(OUTPUT_HTML, "w") as f: f.write(html)
    print(f"🏁 DONE: {OUTPUT_HTML}")

if __name__ == "__main__":
    run_pipeline()


--- [TRACE] LOADING LOOKUPS & GEOSPATIAL LAYERS ---
✅ Excel Lookups Hydrated.
--- [TRACE] GEOSPATIAL JOIN FOR OFFICIAL UTILITIES AND CLIMATE ZONES ---
✅ Spatial Layers and Nearest-Neighbor boundaries resolved.

--- [TRACE] PROCESSING INVENTORY LOOP COUNTY-BY-COUNTY ---
 PROCESSING DATA FILE: Alameda.csv | Locked Target Pool: 001 (Alameda)
   -> FILE TOTALS: Matches: 8,946 | Spatial Fallbacks Saved: 0
 PROCESSING DATA FILE: Alpine.csv | Locked Target Pool: 003 (Alpine)
   -> FILE TOTALS: Matches: 5 | Spatial Fallbacks Saved: 0
 PROCESSING DATA FILE: Amador.csv | Locked Target Pool: 005 (Amador)
   -> FILE TOTALS: Matches: 173 | Spatial Fallbacks Saved: 0
 PROCESSING DATA FILE: Butte.csv | Locked Target Pool: 007 (Butte)
   -> FILE TOTALS: Matches: 1,425 | Spatial Fallbacks Saved: 0
 PROCESSING DATA FILE: Calaveras.csv | Locked Target Pool: 009 (Calaveras)
   -> FILE TOTALS: Matches: 250 | Spatial Fallbacks Saved: 0
 PROCESSING DATA FILE: Colusa.csv | Locked Target Pool: 011 (Colusa)
  

## Bonus Work - Creating Samples for Export ##

In [ ]:
# ==============================================================================
# Original raw sample inputs from Alameda
Alameda_kml_samples_to_export = ["6000104287","6000104277","6000104276","6000104273","6000104278","6000104286"]

# Conversion to map the FIPS data style
corrected_samples = []
for fid in Alameda_kml_samples_to_export:
    clean_id = fid.strip()
    if clean_id.startswith("60001") and len(clean_id) == 10:
        tract_part = clean_id[5:]
        formatted_id = f"06001{int(tract_part)}00"
        corrected_samples.append((clean_id, formatted_id)) # Keep original name for file naming
    else:
        corrected_samples.append((clean_id, clean_id))

# Replicating the geospatial join logic to extract KML samples for Alameda County
# Load Census Tract Spatial Geometry Layer
TRACT_SHP = 'Data Sources/CA_2025_tracts.zip' 
raw_tracts = gpd.read_file(TRACT_SHP).to_crs("EPSG:4269")
raw_tracts['GEOID_STR'] = raw_tracts['GEOID'].astype(str).str.strip()

# Standardize column constraints
raw_tracts['GEOID_BASE'] = raw_tracts['GEOID_STR'].apply(lambda x: x[-11:] if len(x) >= 11 else x.zfill(11))
raw_tracts['NAME_CLEAN'] = raw_tracts['NAME'].apply(clean_sub_tract_label)
raw_tracts['NAMELSAD_CLEAN'] = raw_tracts['NAMELSAD'].apply(lambda x: "Census Tract " + clean_sub_tract_label(str(x).replace("Census Tract ", "")))

agg_rules = {}
for col in raw_tracts.columns:
    if col in ['geometry', 'GEOID_BASE']: continue
    if col in ['ALAND', 'AWATER']: agg_rules[col] = 'sum'
    else: agg_rules[col] = 'first'
        
tracts_gdf = raw_tracts.dissolve(by='GEOID_BASE', aggfunc=agg_rules).reset_index()
all_california_tracts = set(tracts_gdf['GEOID_BASE'].unique())

print("--- [TRACE] GEOSPATIAL JOIN FOR OFFICIAL UTILITIES AND CLIMATE ZONES ---")
cz_gdf = gpd.read_file(CEC_CLIMATE_ZONE_SHP_ZIP).to_crs("EPSG:4269")
util_gdf = gpd.read_file(UTILITY_SHP_ZIP).to_crs("EPSG:4269")

# Loop through each target tract to create isolated visual checks and KML exports
for original_name, fips_id in corrected_samples:
    # Isolate just this single tract row
    single_tract = tracts_gdf[tracts_gdf['GEOID_BASE'] == fips_id]
    
    if single_tract.empty:
        print(f"SKIPPED: No geometry found for tract {original_name} (FIPS: {fips_id})")
        continue
        
    print(f"\nProcessing Tract {original_name}...")

    # Generate an individual static PNG check
    fig, ax = plt.subplots(figsize=(6, 6))
    single_tract.plot(color='seagreen', edgecolor='black', ax=ax, aspect='equal')
    ax.set_title(f"Alameda Tract Check: {original_name}")
    ax.set_axis_off()
    plt.tight_layout()
    
    png_filename = f"Alameda_Tract_{original_name}_Map.png"
    plt.savefig(png_filename, dpi=120)
    plt.close(fig)
    print(f"Saved visual plot: {png_filename}")

    # Export the single tract out to its own KML file
    kml_filename = f"Alameda_Tract_{original_name}.kml"
    single_tract.to_file(kml_filename, driver='KML')
    print(f"Saved isolated KML: {kml_filename}")

--- [TRACE] GEOSPATIAL JOIN FOR OFFICIAL UTILITIES AND CLIMATE ZONES ---

Processing Tract 6000104287...
  ✅ Saved visual plot: Alameda_Tract_6000104287_Map.png
  ✅ Saved isolated KML: Alameda_Tract_6000104287.kml

Processing Tract 6000104277...
  ✅ Saved visual plot: Alameda_Tract_6000104277_Map.png
  ✅ Saved isolated KML: Alameda_Tract_6000104277.kml

Processing Tract 6000104276...
  ✅ Saved visual plot: Alameda_Tract_6000104276_Map.png
  ✅ Saved isolated KML: Alameda_Tract_6000104276.kml

Processing Tract 6000104273...
  ✅ Saved visual plot: Alameda_Tract_6000104273_Map.png
  ✅ Saved isolated KML: Alameda_Tract_6000104273.kml

Processing Tract 6000104278...
  ✅ Saved visual plot: Alameda_Tract_6000104278_Map.png
  ✅ Saved isolated KML: Alameda_Tract_6000104278.kml

Processing Tract 6000104286...
  ✅ Saved visual plot: Alameda_Tract_6000104286_Map.png
  ✅ Saved isolated KML: Alameda_Tract_6000104286.kml


In [ ]:
# ================================================================
# Export of Census Tract data with County Names for verification and QA purposes

import geopandas as gpd

# Corrected mapping: 3-digit Federal County FIPS code -> County Name
FIPS_TO_COUNTY = {
    "001": "Alameda", "003": "Alpine", "005": "Amador", "007": "Butte", "009": "Calaveras",
    "011": "Colusa", "013": "Contra Costa", "015": "Del Norte", "017": "El Dorado", "019": "Fresno",
    "021": "Glenn", "023": "Humboldt", "025": "Imperial", "027": "Inyo", "029": "Kern",
    "031": "Kings", "033": "Lake", "035": "Lassen", "037": "Los Angeles", "039": "Madera",
    "041": "Marin", "043": "Mariposa", "045": "Mendocino", "047": "Merced", "049": "Modoc",
    "051": "Mono", "053": "Monterey", "055": "Napa", "057": "Nevada", "059": "Orange",
    "061": "Placer", "063": "Plumas", "065": "Riverside", "067": "Sacramento", "069": "San Benito",
    "071": "San Bernardino", "073": "San Diego", "075": "San Francisco", "077": "San Joaquin",
    "079": "San Luis Obispo", "081": "San Mateo", "083": "Santa Barbara", "085": "Santa Clara",
    "087": "Santa Cruz", "089": "Shasta", "091": "Sierra", "093": "Siskiyou", "095": "Solano",
    "097": "Sonoma", "099": "Stanislaus", "101": "Sutter", "103": "Tehama", "105": "Trinity",
    "107": "Tulare", "109": "Tuolumne", "111": "Ventura", "113": "Yolo", "115": "Yuba"
}

# ==============================================================================
# Limited export of Census Tract data for verification and QA
ca_census_gdf = gpd.read_file(TRACT_SHP).to_crs("EPSG:4269")

# Appending the County where the census tract is located for easier using a spatial join 
ca_county_gdf = gpd.read_file("Data Sources/ca_counties.zip").to_crs("EPSG:4269")
ca_census_gdf = gpd.sjoin(ca_census_gdf, ca_county_gdf[['COUNTYFP', 'geometry']], how='left', predicate='intersects')

# Ensure the column is a 3-digit zero-padded string to safely match the dictionary keys
ca_census_gdf['County_Name'] = ca_census_gdf['COUNTYFP_right'].astype(str).str.zfill(3).map(FIPS_TO_COUNTY) 

# Save data from the gdf to a CSV file for verification
ca_census_gdf.drop(columns='geometry').to_csv("CA_Census_Tracts_With_Counties.csv", index=False)

In [ ]:
# (7) Interactive HTML of County-level aggregates, toggle-able to allow for variable approaches to conversion of HVAC tonnage
# ==============================================================================
# Configuration section - likely to be updated with results from the new TRC study!
default_tonnage_weight = 0.3      # Default slider value to scale the combined tonnage
avg_rtu_tonnage = 8.9             # Configurable average unit size in tons to back-calculate RTUs
OUTPUT_COUNTY_HTML = "County_Level_WEIGHTED_COMStock_Summary.html"
OUTPUT_COUNTY_CSV = "County_Level_WEIGHTED_COMStock_Summary.csv"

# ==============================================================================
# County data creation
print("--- California Spatial Layer Re-Construction ---")
redo_ca_county_gdf = gpd.read_file("Data Sources/ca_counties.zip").to_crs("EPSG:4269") # make sure this is the best CRS

# Catching the variety of spellings for the county column (makes this code more adaptable to new sources)
possible_name_cols = ['NAME', 'CountyName', 'COUNTY', 'county_nam', 'CO_NAME', 'County_Name']
shapefile_name_col = None
for col in possible_name_cols:
    if col in redo_ca_county_gdf.columns:
        shapefile_name_col = col
        break

if shapefile_name_col is None:
    object_cols = [c for c in redo_ca_county_gdf.columns if c != 'geometry']
    shapefile_name_col = object_cols[0] if object_cols else redo_ca_county_gdf.columns[0]

redo_ca_county_gdf['County_Name_Clean'] = (
    redo_ca_county_gdf[shapefile_name_col]
    .astype(str)
    .str.replace(r'(?i)\s*County$', '', regex=True)
    .str.strip()
    .str.title()
)

# Re-loading the tract level data
print("--- Re-loading the Data from Step 6 ---")
census_with_county_df = pd.read_csv("/Users/neil.stein/Documents/CFB/Western Campaigns/AC2HP/Tract_Level_COMStock_Full_Statewide_Final_Census_Tract_ANALYSIS_11.113.csv")

census_with_county_df['County_Name_Clean'] = (
    census_with_county_df['County_Name']
    .astype(str)
    .str.replace(r'(?i)\s*County$', '', regex=True)
    .str.strip()
    .str.title()
)

# Odd bug found - the previous code block seems to have mislabeled the Stanislaus County data on export (but oddly the map there is fine)
# This is a hard-coded fix, drop it for other code runs
census_with_county_df['County_Name_Clean'] = census_with_county_df['County_Name_Clean'].replace({'Ca': 'Stanislaus'})

# Aggregating county stats now that we have standardized and re-loaded the data
print("--- County Level Data Aggregation ---")

# First, roll up the clean sector-specific metrics by summing the tracts
county_summary = census_with_county_df.groupby('County_Name_Clean').agg({
    'CEC_Climate_Zone': lambda x: ', '.join(sorted(x.dropna().astype(str).str.strip().unique())),
    'Acronym': lambda x: ', '.join(sorted(x.dropna().astype(str).str.strip().unique())),
    're_tons': 'sum', 'so_tons': 'sum',
    're_kwh': 'sum', 'so_kwh': 'sum',
    're_therms': 'sum', 'so_therms': 'sum',
    're_ghg': 'sum', 'so_ghg': 'sum',
    're_npv': 'sum', 'so_npv': 'sum'
}).reset_index()

# Re-creating the combined impacts on the county scale in case of errors with split tracts across counties
county_summary['co_tons'] = county_summary['re_tons'] + county_summary['so_tons']
county_summary['co_kwh'] = county_summary['re_kwh'] + county_summary['so_kwh']
county_summary['co_therms'] = county_summary['re_therms'] + county_summary['so_therms']
county_summary['co_ghg'] = county_summary['re_ghg'] + county_summary['so_ghg']
county_summary['co_npv'] = county_summary['re_npv'] + county_summary['so_npv']

# Reorder columns to match original spec
column_order = [
    'County_Name_Clean', 'CEC_Climate_Zone', 'Acronym',
    're_tons', 'so_tons', 'co_tons',
    're_kwh', 'so_kwh', 'co_kwh',
    're_therms', 'so_therms', 'co_therms',
    're_ghg', 'so_ghg', 'co_ghg',
    're_npv', 'so_npv', 'co_npv'
]
county_summary = county_summary[column_order]

county_summary.to_csv(OUTPUT_COUNTY_CSV, index=False)
print(f"Exported county-level summary stats to: {OUTPUT_COUNTY_CSV}")

# Creating a choropleth and merging all the data to the base layer
merged_county_gdf = redo_ca_county_gdf.merge(county_summary, on='County_Name_Clean', how='inner')
print(f"Successfully joined {len(merged_county_gdf)} counties for geographic display.")

# Calculate initial weighted tonnage
default_totals = merged_county_gdf['co_tons'] * default_tonnage_weight
valid_totals = default_totals.dropna()

# Sorting tonnage into bins
if not valid_totals.empty and valid_totals.max() > 0:
    map_bins = list(np.nanquantile(valid_totals, [0.25, 0.50, 0.75, 0.95]))
else:
    map_bins = [1000, 5000, 20000, 100000]

map_bins = [float(x) if (np.isfinite(x) and not np.isnan(x)) else 0.0 for x in map_bins]
geojson_data = json.loads(merged_county_gdf.to_json())

# ==============================================================================
# HTML code for building out the map
# Some small changes can be made here to the labels, rounding of aggregated stats, etc. 

html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <title>County Level COMStock Summary Map</title>
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
    <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
    <style>
        body {{ margin: 0; padding: 0; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif; }}
        #map {{ height: 100vh; background: #f4f4f4; }}
        .control-panel {{
            position: absolute;
            top: 20px;
            right: 20px;
            background: white;
            padding: 15px;
            border-radius: 8px;
            box-shadow: 0 4px 12px rgba(0,0,0,0.15);
            z-index: 1000;
            width: 300px;
        }}
        .control-panel h3 {{ margin: 0 0 5px 0; font-size: 14px; color: #111; }}
        .control-panel p {{ font-size: 11px; color: #666; margin: 0 0 12px 0; line-height: 1.4; }}
        .slider-container {{ display: flex; align-items: center; justify-content: space-between; }}
        .slider-container input {{ width: 75%; cursor: pointer; }}
        .slider-val {{ font-weight: bold; font-size: 14px; color: #238b45; min-width: 35px; text-align: right; }}
    </style>
</head>
<body>

<div class="control-panel">
    <h3>Combined Tonnage Weight Factor</h3>
    <p>Slide to dynamically adjust the estimated share of <b>5-20 ton units</b> across all counties. Baseline (30%) based on historical AHRI Shipment data.</p>
    <div class="slider-container">
        <input type="range" id="weightSlider" min="0.0" max="1.0" step="0.05" value="{default_tonnage_weight}">
        <span id="weightDisplay" class="slider-val">{default_tonnage_weight:.2f}</span>
    </div>
</div>

<div id="map"></div>

<script>
    var map = L.map('map').setView([37.2, -119.7], 6);
    
    L.tileLayer('https://{{s}}.basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}.png', {{
        attribution: '&copy; OpenStreetMap contributors &copy; CartoDB'
    }}).addTo(map);

    var currentWeight = {default_tonnage_weight};
    var avgRtuSize = {avg_rtu_tonnage};
    var mapBins = {json.dumps(map_bins)};
    var geojsonRaw = {json.dumps(geojson_data)};
    var countyLayer;

    // Direct simple multiplication of the combined impact tonnage
    function calculateWeightedTons(props, weight) {{
        return (props.co_tons || 0) * weight;
    }}

    function getColor(v) {{
        return v > mapBins[3] ? '#00441b' : 
               v > mapBins[2] ? '#238b45' : 
               v > mapBins[1] ? '#74c476' : '#e5f5e0';
    }}

    function styleFeature(feature) {{
        var value = calculateWeightedTons(feature.properties, currentWeight);
        return {{
            fillColor: getColor(value),
            weight: 1.2,
            color: '#ffffff',
            fillOpacity: 0.85
        }};
    }}

    function updateTooltips(layer) {{
        layer.eachLayer(function(l) {{
            var p = l.feature.properties;
            
            // Calculate slider-affected metrics for all fields dynamically with the toggle 

            var w_re_tons = (p.re_tons || 0) * currentWeight;
            var w_so_tons = (p.so_tons || 0) * currentWeight;
            var w_co_tons = (p.co_tons || 0) * currentWeight;
            
            var w_re_kwh = (p.re_kwh || 0) * currentWeight * -1; // Invert kWh calculations since the current label describes them as increased demand
            var w_so_kwh = (p.so_kwh || 0) * currentWeight * -1; // feel free to flip the label to display the values as written
            var w_co_kwh = (p.co_kwh || 0) * currentWeight * -1;
            
            var w_re_therms = (p.re_therms || 0) * currentWeight;
            var w_so_therms = (p.so_therms || 0) * currentWeight;
            var w_co_therms = (p.co_therms || 0) * currentWeight;
            
            var w_re_ghg = (p.re_ghg || 0) * currentWeight;
            var w_so_ghg = (p.so_ghg || 0) * currentWeight;
            var w_co_ghg = (p.co_ghg || 0) * currentWeight;
            
            var w_re_npv = (p.re_npv || 0) * currentWeight;
            var w_so_npv = (p.so_npv || 0) * currentWeight;
            var w_co_npv = (p.co_npv || 0) * currentWeight;

            // Back-calculate the estimate of RTUs dynamically using the weighted profile

            var estimatedRTUs = w_co_tons > 0 ? Math.round(w_co_tons / avgRtuSize) : 0;
            
            l.bindTooltip(`
                <div style="font-size:12px; min-width:290px; padding: 4px;">
                    <strong style="font-size:15px; color:#111;">${{p.County_Name_Clean}} County Summary</strong><br>
                    <span style="font-size:11px; color:#555;">
                        <b>CZs Present:</b> ${{p.CEC_Climate_Zone || 'None Reported'}}<br>
                        <b>Utilities Serving:</b> ${{p.Acronym || 'None Reported'}}
                    </span><hr style="border:0; border-top:1px solid #ddd; margin: 8px 0;">
                    
                    <table style="width:100%; border-collapse: collapse; font-size:11px;">
                        <tr style="border-bottom: 2px solid #bbb; font-weight:bold; text-align:left; color:#333;">
                            <td style="padding:2px 0;">Sector Metric</td>
                            <td>Retail</td>
                            <td>Small Office</td>
                            <td>Combined</td>
                        </tr>
                        <tr style="border-bottom: 1px solid #eee;">
                            <td style="padding:4px 0; font-weight:bold;">HVAC Capacity (Tons):</td>
                            <td>${{Math.round(w_re_tons).toLocaleString()}} t</td>
                            <td>${{Math.round(w_so_tons).toLocaleString()}} t</td>
                            <td>${{Math.round(w_co_tons).toLocaleString()}} t</td>
                        </tr>
                        <tr style="border-bottom: 1px solid #eee;">
                            <td style="padding:4px 0; font-weight:bold;">Electricity Increase (kWh):</td> // change the kWh label here!
                            <td>${{Math.round(w_re_kwh).toLocaleString()}}</td>
                            <td>${{Math.round(w_so_kwh).toLocaleString()}}</td>
                            <td>${{Math.round(w_co_kwh).toLocaleString()}}</td>
                        </tr>
                        <tr style="border-bottom: 1px solid #eee;">
                            <td style="padding:4px 0; font-weight:bold;">Natural Gas Reduced (Therms):</td>
                            <td>${{Math.round(w_re_therms).toLocaleString()}}</td>
                            <td>${{Math.round(w_so_therms).toLocaleString()}}</td>
                            <td>${{Math.round(w_co_therms).toLocaleString()}}</td>
                        </tr>
                        <tr style="border-bottom: 1px solid #eee;">
                            <td style="padding:4px 0; font-weight:bold;">GHG Reduction (Tons):</td>
                            <td>${{Math.round(w_re_ghg).toLocaleString()}}</td>
                            <td>${{Math.round(w_so_ghg).toLocaleString()}}</td>
                            <td>${{Math.round(w_co_ghg).toLocaleString()}}</td>
                        </tr>
                        <tr>
                            <td style="padding:4px 0; font-weight:bold;">Financial NPV:</td>
                            <td>$${{Math.round(w_re_npv).toLocaleString()}}</td>
                            <td>$${{Math.round(w_so_npv).toLocaleString()}}</td>
                            <td>$${{Math.round(w_co_npv).toLocaleString()}}</td>
                        </tr>
                    </table><hr style="border:0; border-top:1px solid #ddd; margin: 8px 0;">
                    
                    <div style="background:#f6f9f6; padding:8px; border-radius:5px; border:1px solid #d0e2d0; margin-bottom: 6px;">
                        <span style="font-size:11px; color:#555; font-weight:bold; display:block; margin-bottom:2px;">
                            CURRENT WEIGHTED METRIC PROFILE:
                        </span>
                        <span style="font-size:14px; color:#00441b; font-weight:bold;">
                            ${{Math.round(w_co_tons).toLocaleString()}} Weighted Combined Tons
                        </span>
                        <span style="font-size:10px; color:#666; display:block; margin-top:2px; font-style:italic;">
                            Formula: Combined Impact &times; ${{currentWeight.toFixed(2)}}
                        </span>
                    </div>

                    <div style="background:#f4f7fa; padding:8px; border-radius:5px; border:1px solid #ccd9e8;">
                        <span style="font-size:11px; color:#444; font-weight:bold; display:block; margin-bottom:2px;">
                            EQUIVALENT ASSET ESTIMATION:
                        </span>
                        <span style="font-size:13px; color:#1a5276; font-weight:bold;">
                            ${{estimatedRTUs.toLocaleString()}} Estimated RTUs
                        </span>
                        <span style="font-size:10px; color:#666; display:block; margin-top:2px; font-style:italic;">
                            Based on an average size of ${{avgRtuSize}} tons per unit
                        </span>
                    </div>
                </div>`, {{ sticky: true }});
        }});
    }}

    countyLayer = L.geoJson(geojsonRaw, {{
        style: styleFeature
    }}).addTo(map);
    
    updateTooltips(countyLayer);

    document.getElementById('weightSlider').addEventListener('input', function(e) {{
        currentWeight = parseFloat(e.target.value);
        document.getElementById('weightDisplay').innerText = currentWeight.toFixed(2);
        
        countyLayer.setStyle(styleFeature);
        updateTooltips(countyLayer);
    }});
</script>
</body>
</html>
"""

with open(OUTPUT_COUNTY_HTML, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"HTML Interactive county summary generated as {OUTPUT_COUNTY_HTML}")

--- [TRACE] LOADING CALIFORNIA COUNTY SPATIAL LAYER ---
--- [TRACE] LOADING TRACT-LEVEL CENSUS ANALYSIS SOURCE DATA ---
--- [TRACE] RUNNING STRATEGIC DATA AGGREGATION ---
✅ Exported flat summary stats to: County_Level_WEIGHTED_COMStock_Summary.csv
Successfully joined 58 counties for geographic display.
🏁 DONE: Interactive county summary generated at County_Level_WEIGHTED_COMStock_Summary.html


In [11]:
# San Mateo Sample Export for Sourcing Geospatial Imagery

import geopandas as gpd
import matplotlib.pyplot as plt

def clean_sub_tract_label(label):
    return str(label).strip()

# ==============================================================================
# Target San Mateo Raw Inputs
san_mateo_kml_samples_to_export = ["6008106063", "6008106077"]

# Map raw inputs to match your shapefile's format: "06081" + "606300"
corrected_samples = []
for fid in san_mateo_kml_samples_to_export:
    clean_id = fid.strip()
    if len(clean_id) == 10:
        # Extract the tract digits at the end (e.g., "06063")
        tract_digits = clean_id[5:] 
        # Convert to int to drop the leading zero, turning "06063" -> 6063. 
        # Then format it to match the file's 6-digit tract suffix layout ("606300")
        formatted_id = f"06081{int(tract_digits)}00"
        corrected_samples.append((clean_id, formatted_id))
    else:
        corrected_samples.append((clean_id, clean_id))

# Load Census Tract Spatial Geometry Layer
TRACT_SHP = 'Data Sources/CA_2025_tracts.zip' 
raw_tracts = gpd.read_file(TRACT_SHP).to_crs("EPSG:4269")
raw_tracts['GEOID_STR'] = raw_tracts['GEOID'].astype(str).str.strip()

# Standardize column constraints
raw_tracts['GEOID_BASE'] = raw_tracts['GEOID_STR'].apply(lambda x: x[-11:] if len(x) >= 11 else x.zfill(11))
raw_tracts['NAME_CLEAN'] = raw_tracts['NAME'].apply(clean_sub_tract_label)
raw_tracts['NAMELSAD_CLEAN'] = raw_tracts['NAMELSAD'].apply(lambda x: "Census Tract " + clean_sub_tract_label(str(x).replace("Census Tract ", "")))

agg_rules = {}
for col in raw_tracts.columns:
    if col in ['geometry', 'GEOID_BASE']: continue
    if col in ['ALAND', 'AWATER']: agg_rules[col] = 'sum'
    else: agg_rules[col] = 'first'
        
tracts_gdf = raw_tracts.dissolve(by='GEOID_BASE', aggfunc=agg_rules).reset_index()

# ------------------------------------------------------------------------------
# Loop through each target tract to create isolated visual checks and KML exports
# ------------------------------------------------------------------------------
for original_name, fips_id in corrected_samples:
    # Isolate just this single tract row
    single_tract = tracts_gdf[tracts_gdf['GEOID_BASE'] == fips_id]
    
    if single_tract.empty:
        print(f"⚠️ SKIPPED: No geometry found for tract {original_name} (FIPS: {fips_id})")
        continue
        
    print(f"\nProcessing Tract {original_name} (Mapped to FIPS: {fips_id})...")

    # 1. Generate an individual static PNG check
    fig, ax = plt.subplots(figsize=(6, 6))
    single_tract.plot(color='seagreen', edgecolor='black', ax=ax, aspect='equal')
    ax.set_title(f"San Mateo Tract Check: {original_name}")
    ax.set_axis_off()
    plt.tight_layout()
    
    png_filename = f"San_Mateo_Tract_{original_name}_Map.png"
    plt.savefig(png_filename, dpi=120)
    plt.close(fig) # Free memory
    print(f"  ✅ Saved visual plot: {png_filename}")

    # 2. Export the single tract out to its own KML file
    kml_filename = f"San_Mateo_Tract_{original_name}.kml"
    single_tract.to_file(kml_filename, driver='KML')
    print(f"  ✅ Saved isolated KML: {kml_filename}")


Processing Tract 6008106063 (Mapped to FIPS: 06081606300)...
  ✅ Saved visual plot: San_Mateo_Tract_6008106063_Map.png
  ✅ Saved isolated KML: San_Mateo_Tract_6008106063.kml
⚠️ SKIPPED: No geometry found for tract 6008106077 (FIPS: 06081607700)
